In [1]:
# ============================================
# Cell 1: Imports, config, paths, utilities
# ============================================

import os
import gc
import json
import math
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# -----------------------------
# Global setup
# -----------------------------
ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
PROCESSED_ROOT = ROOT / "data" / "processed"
SPLIT_ROOT = ROOT / "data" / "splits"

RESULTS_ROOT = ROOT / "results"
TABLES_ROOT = RESULTS_ROOT / "tables"
LOGS_ROOT = RESULTS_ROOT / "logs"
CKPT_ROOT = RESULTS_ROOT / "checkpoints"
PRED_ROOT = RESULTS_ROOT / "predictions"
HIST_ROOT = RESULTS_ROOT / "histories"

for p in [TABLES_ROOT, LOGS_ROOT, CKPT_ROOT, PRED_ROOT, HIST_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# -----------------------------
# Reproducibility
# -----------------------------
GLOBAL_SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(GLOBAL_SEED)

# -----------------------------
# Dataset list and run config
# -----------------------------
DATASET_RUN_CONFIG = {
    "etth1": {
        "seq_len": 96,
        "pred_len": 96,
        "seasonality": 24,
        "batch_size": 256,
        "epochs": 20,
    },
    "etth2": {
        "seq_len": 96,
        "pred_len": 96,
        "seasonality": 24,
        "batch_size": 256,
        "epochs": 20,
    },
    "ettm1": {
        "seq_len": 192,
        "pred_len": 96,
        "seasonality": 96,
        "batch_size": 256,
        "epochs": 20,
    },
    "ettm2": {
        "seq_len": 192,
        "pred_len": 96,
        "seasonality": 96,
        "batch_size": 256,
        "epochs": 20,
    },
    "weather": {
        "seq_len": 288,
        "pred_len": 96,
        "seasonality": 144,
        "batch_size": 192,
        "epochs": 20,
    },
    "electricity": {
        "seq_len": 168,
        "pred_len": 96,
        "seasonality": 24,
        "batch_size": 128,
        "epochs": 20,
    },
    "traffic": {
        "seq_len": 168,
        "pred_len": 96,
        "seasonality": 24,
        "batch_size": 96,
        "epochs": 20,
    },
    "exchange": {
        "seq_len": 96,
        "pred_len": 96,
        "seasonality": 7,
        "batch_size": 256,
        "epochs": 20,
    },
    "ili": {
        "seq_len": 104,
        "pred_len": 24,
        "seasonality": 52,
        "batch_size": 64,
        "epochs": 30,
    },
}

PHASE_A_MODELS = [
    "naive",
    "seasonal_naive",
    "lstm",
    "gru",
    "dlinear",
    "fixed_dwt_lstm",
]

# -----------------------------
# Training config
# -----------------------------
TRAIN_CFG = {
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "patience": 5,
    "num_workers": 2,
    "use_amp": torch.cuda.is_available(),
    "dropout": 0.1,
    "hidden_dim": 128,
    "num_layers": 2,
    "proj_dim": 128,
    "dlinear_kernel": 25,
}

# -----------------------------
# Utilities
# -----------------------------
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def safe_json_dump(obj, path):
    def convert(x):
        if isinstance(x, (np.integer,)):
            return int(x)
        if isinstance(x, (np.floating,)):
            return float(x)
        if isinstance(x, (np.ndarray,)):
            return x.tolist()
        return x
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=convert)

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mse = np.mean((y_true - y_pred) ** 2)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = float(np.sqrt(mse))
    return {
        "mse": float(mse),
        "mae": float(mae),
        "rmse": rmse,
    }

def seasonal_repeat(last_cycle, pred_len):
    last_cycle = np.asarray(last_cycle)
    rep = int(np.ceil(pred_len / len(last_cycle)))
    out = np.tile(last_cycle, rep)[:pred_len]
    return out

def load_prepared_dataset(dataset_name):
    npz_path = PROCESSED_ROOT / dataset_name / f"{dataset_name}_prepared.npz"
    meta_path = PROCESSED_ROOT / dataset_name / f"{dataset_name}_meta.json"
    split_path = SPLIT_ROOT / f"{dataset_name}_splits.json"

    if not npz_path.exists():
        raise FileNotFoundError(f"Missing: {npz_path}")
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing: {meta_path}")
    if not split_path.exists():
        raise FileNotFoundError(f"Missing: {split_path}")

    data = np.load(npz_path, allow_pickle=True)
    with open(meta_path, "r") as f:
        meta = json.load(f)
    with open(split_path, "r") as f:
        splits = json.load(f)

    bundle = {
        "values_raw": data["values_raw"].astype(np.float32),
        "values_scaled": data["values_scaled"].astype(np.float32),
        "time_features": data["time_features"].astype(np.float32),
        "dates": data["dates"],
        "scaler_mean": data["scaler_mean"].astype(np.float32),
        "scaler_std": data["scaler_std"].astype(np.float32),
        "target_idx": int(data["target_idx"][0]),
        "meta": meta,
        "splits": {k: int(v) for k, v in splits.items()},
    }
    return bundle

print("ROOT:", ROOT)
print("Phase A models:", PHASE_A_MODELS)
print("Datasets:", list(DATASET_RUN_CONFIG.keys()))

DEVICE: cuda
GPU: NVIDIA H100 NVL
ROOT: /data/Sajjan_Singh/spml/wavelet_seq_project
Phase A models: ['naive', 'seasonal_naive', 'lstm', 'gru', 'dlinear', 'fixed_dwt_lstm']
Datasets: ['etth1', 'etth2', 'ettm1', 'ettm2', 'weather', 'electricity', 'traffic', 'exchange', 'ili']


In [2]:
# ============================================
# Cell 2: Dataset windows, models, training/eval
# ============================================

class ForecastWindowDataset(Dataset):
    def __init__(self, bundle, split_name, seq_len, pred_len):
        self.values_raw = bundle["values_raw"]
        self.values_scaled = bundle["values_scaled"]
        self.target_idx = bundle["target_idx"]
        self.splits = bundle["splits"]
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.split_name = split_name

        train_start = self.splits["train_start"]
        train_end = self.splits["train_end"]
        val_start = self.splits["val_start"]
        val_end = self.splits["val_end"]
        test_start = self.splits["test_start"]
        test_end = self.splits["test_end"]

        if split_name == "train":
            self.border1 = train_start
            self.border2 = train_end
        elif split_name == "val":
            self.border1 = max(0, val_start - seq_len)
            self.border2 = val_end
        elif split_name == "test":
            self.border1 = max(0, test_start - seq_len)
            self.border2 = test_end
        else:
            raise ValueError(f"Invalid split_name: {split_name}")

        self.length = self.border2 - self.border1 - seq_len - pred_len + 1
        if self.length <= 0:
            raise ValueError(
                f"Dataset length <= 0 for split={split_name}, "
                f"border1={self.border1}, border2={self.border2}, "
                f"seq_len={seq_len}, pred_len={pred_len}"
            )

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        s = self.border1 + idx
        seq_x_scaled = self.values_scaled[s : s + self.seq_len]                        # [L, D]
        seq_y_scaled = self.values_scaled[s + self.seq_len : s + self.seq_len + self.pred_len, self.target_idx]  # [P]
        seq_x_target_raw = self.values_raw[s : s + self.seq_len, self.target_idx]      # [L]
        seq_y_raw = self.values_raw[s + self.seq_len : s + self.seq_len + self.pred_len, self.target_idx]        # [P]

        return {
            "x_scaled": torch.tensor(seq_x_scaled, dtype=torch.float32),
            "y_scaled": torch.tensor(seq_y_scaled, dtype=torch.float32),
            "x_target_raw": torch.tensor(seq_x_target_raw, dtype=torch.float32),
            "y_raw": torch.tensor(seq_y_raw, dtype=torch.float32),
        }

def make_loaders(bundle, seq_len, pred_len, batch_size, num_workers=2):
    train_ds = ForecastWindowDataset(bundle, "train", seq_len, pred_len)
    val_ds = ForecastWindowDataset(bundle, "val", seq_len, pred_len)
    test_ds = ForecastWindowDataset(bundle, "test", seq_len, pred_len)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=False
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=False
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=False
    )
    return train_loader, val_loader, test_loader

# -----------------------------
# Models
# -----------------------------
class LSTMForecast(nn.Module):
    def __init__(self, input_dim, pred_len, proj_dim=128, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.rnn = nn.LSTM(
            input_size=proj_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, pred_len),
        )

    def forward(self, x):
        x = self.proj(x)
        out, _ = self.rnn(x)
        h = out[:, -1, :]
        pred = self.head(h)
        return pred

class GRUForecast(nn.Module):
    def __init__(self, input_dim, pred_len, proj_dim=128, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.rnn = nn.GRU(
            input_size=proj_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, pred_len),
        )

    def forward(self, x):
        x = self.proj(x)
        out, _ = self.rnn(x)
        h = out[:, -1, :]
        pred = self.head(h)
        return pred

class MovingAverage(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        # x: [B, L, D]
        pad = (self.kernel_size - 1) // 2
        front = x[:, 0:1, :].repeat(1, pad, 1)
        end = x[:, -1:, :].repeat(1, pad, 1)
        x_pad = torch.cat([front, x, end], dim=1)      # [B, L+2pad, D]
        x_pad = x_pad.permute(0, 2, 1)                 # [B, D, L+2pad]
        ma = self.avg(x_pad).permute(0, 2, 1)          # [B, L, D]
        return ma

class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAverage(kernel_size)

    def forward(self, x):
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend

class DLinearForecast(nn.Module):
    def __init__(self, seq_len, pred_len, target_idx, kernel_size=25):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.target_idx = target_idx
        self.decomp = SeriesDecomposition(kernel_size=kernel_size)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)
        self.linear_trend = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        # x: [B, L, D]
        seasonal, trend = self.decomp(x)
        seasonal = seasonal.permute(0, 2, 1)   # [B, D, L]
        trend = trend.permute(0, 2, 1)         # [B, D, L]

        seasonal_out = self.linear_seasonal(seasonal)  # [B, D, P]
        trend_out = self.linear_trend(trend)           # [B, D, P]
        out = seasonal_out + trend_out                 # [B, D, P]

        pred = out[:, self.target_idx, :]              # [B, P]
        return pred

def fixed_haar_features(x):
    # x: [B, L, D]
    B, L, D = x.shape
    if L % 2 == 1:
        x = torch.cat([x, x[:, -1:, :]], dim=1)
        L = L + 1

    even = x[:, 0::2, :]
    odd = x[:, 1::2, :]

    low = (even + odd) / math.sqrt(2.0)
    high = (even - odd) / math.sqrt(2.0)

    low_up = low.repeat_interleave(2, dim=1)[:, :L, :]
    high_up = high.repeat_interleave(2, dim=1)[:, :L, :]

    return low_up, high_up

class FixedDWTLSTMForecast(nn.Module):
    def __init__(self, input_dim, pred_len, proj_dim=128, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        total_dim = input_dim * 3  # original + low + high
        self.proj = nn.Linear(total_dim, proj_dim)
        self.rnn = nn.LSTM(
            input_size=proj_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, pred_len),
        )

    def forward(self, x):
        low, high = fixed_haar_features(x)
        if low.shape[1] != x.shape[1]:
            low = low[:, :x.shape[1], :]
            high = high[:, :x.shape[1], :]
        x_cat = torch.cat([x, low, high], dim=-1)
        x_cat = self.proj(x_cat)
        out, _ = self.rnn(x_cat)
        h = out[:, -1, :]
        pred = self.head(h)
        return pred

def build_model(model_name, input_dim, seq_len, pred_len, target_idx):
    if model_name == "lstm":
        return LSTMForecast(
            input_dim=input_dim,
            pred_len=pred_len,
            proj_dim=TRAIN_CFG["proj_dim"],
            hidden_dim=TRAIN_CFG["hidden_dim"],
            num_layers=TRAIN_CFG["num_layers"],
            dropout=TRAIN_CFG["dropout"],
        )
    elif model_name == "gru":
        return GRUForecast(
            input_dim=input_dim,
            pred_len=pred_len,
            proj_dim=TRAIN_CFG["proj_dim"],
            hidden_dim=TRAIN_CFG["hidden_dim"],
            num_layers=TRAIN_CFG["num_layers"],
            dropout=TRAIN_CFG["dropout"],
        )
    elif model_name == "dlinear":
        return DLinearForecast(
            seq_len=seq_len,
            pred_len=pred_len,
            target_idx=target_idx,
            kernel_size=TRAIN_CFG["dlinear_kernel"],
        )
    elif model_name == "fixed_dwt_lstm":
        return FixedDWTLSTMForecast(
            input_dim=input_dim,
            pred_len=pred_len,
            proj_dim=TRAIN_CFG["proj_dim"],
            hidden_dim=TRAIN_CFG["hidden_dim"],
            num_layers=TRAIN_CFG["num_layers"],
            dropout=TRAIN_CFG["dropout"],
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")

# -----------------------------
# Classical baselines
# -----------------------------
@torch.no_grad()
def evaluate_naive_baseline(loader):
    preds_all, trues_all = [], []
    for batch in loader:
        x_target_raw = batch["x_target_raw"].numpy()  # [B, L]
        y_raw = batch["y_raw"].numpy()                # [B, P]
        pred = np.repeat(x_target_raw[:, -1:], y_raw.shape[1], axis=1)
        preds_all.append(pred)
        trues_all.append(y_raw)

    preds_all = np.concatenate(preds_all, axis=0)
    trues_all = np.concatenate(trues_all, axis=0)
    metrics = compute_metrics(trues_all.reshape(-1), preds_all.reshape(-1))
    return metrics, preds_all, trues_all

@torch.no_grad()
def evaluate_seasonal_naive_baseline(loader, seasonality):
    preds_all, trues_all = [], []
    for batch in loader:
        x_target_raw = batch["x_target_raw"].numpy()  # [B, L]
        y_raw = batch["y_raw"].numpy()                # [B, P]
        B, L = x_target_raw.shape
        P = y_raw.shape[1]

        pred_batch = []
        for i in range(B):
            use_season = min(seasonality, L)
            last_cycle = x_target_raw[i, -use_season:]
            pred_i = seasonal_repeat(last_cycle, P)
            pred_batch.append(pred_i)

        pred_batch = np.stack(pred_batch, axis=0)
        preds_all.append(pred_batch)
        trues_all.append(y_raw)

    preds_all = np.concatenate(preds_all, axis=0)
    trues_all = np.concatenate(trues_all, axis=0)
    metrics = compute_metrics(trues_all.reshape(-1), preds_all.reshape(-1))
    return metrics, preds_all, trues_all

# -----------------------------
# Neural train/eval
# -----------------------------
def train_one_epoch(model, loader, optimizer, scaler, target_mean, target_std):
    model.train()
    loss_meter = 0.0
    count = 0
    criterion = nn.MSELoss()

    for batch in loader:
        x = batch["x_scaled"].to(DEVICE, non_blocking=True)
        y = batch["y_scaled"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
            pred = model(x)
            loss = criterion(pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = x.size(0)
        loss_meter += loss.item() * bs
        count += bs

    return loss_meter / max(count, 1)

@torch.no_grad()
def evaluate_neural_model(model, loader, target_mean, target_std):
    model.eval()
    criterion = nn.MSELoss()

    loss_meter = 0.0
    count = 0
    preds_all = []
    trues_all = []

    for batch in loader:
        x = batch["x_scaled"].to(DEVICE, non_blocking=True)
        y_scaled = batch["y_scaled"].to(DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=TRAIN_CFG["use_amp"]):
            pred_scaled = model(x)
            loss = criterion(pred_scaled, y_scaled)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean

        bs = x.size(0)
        loss_meter += loss.item() * bs
        count += bs

        preds_all.append(pred_raw)
        trues_all.append(y_raw)

    preds_all = np.concatenate(preds_all, axis=0)
    trues_all = np.concatenate(trues_all, axis=0)

    raw_metrics = compute_metrics(trues_all.reshape(-1), preds_all.reshape(-1))
    scaled_loss = loss_meter / max(count, 1)

    return {
        "scaled_mse_loss": float(scaled_loss),
        "mse": raw_metrics["mse"],
        "mae": raw_metrics["mae"],
        "rmse": raw_metrics["rmse"],
    }, preds_all, trues_all

def train_with_early_stopping(
    model_name,
    model,
    dataset_name,
    train_loader,
    val_loader,
    target_mean,
    target_std,
    epochs,
):
    ckpt_dir = CKPT_ROOT / dataset_name
    hist_dir = HIST_ROOT / dataset_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    hist_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = ckpt_dir / f"{model_name}.pt"
    hist_path = hist_dir / f"{model_name}_history.json"

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_CFG["lr"],
        weight_decay=TRAIN_CFG["weight_decay"]
    )
    scaler = torch.cuda.amp.GradScaler(enabled=TRAIN_CFG["use_amp"])

    best_val = float("inf")
    best_epoch = -1
    patience_left = TRAIN_CFG["patience"]

    history = []

    t0 = time.time()

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, target_mean, target_std)
        val_metrics, _, _ = evaluate_neural_model(model, val_loader, target_mean, target_std)

        record = {
            "epoch": epoch,
            "train_scaled_mse": float(train_loss),
            "val_scaled_mse": float(val_metrics["scaled_mse_loss"]),
            "val_mse_raw": float(val_metrics["mse"]),
            "val_mae_raw": float(val_metrics["mae"]),
            "val_rmse_raw": float(val_metrics["rmse"]),
        }
        history.append(record)

        improved = val_metrics["mse"] < best_val
        if improved:
            best_val = val_metrics["mse"]
            best_epoch = epoch
            patience_left = TRAIN_CFG["patience"]
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_left -= 1

        print(
            f"[{dataset_name} | {model_name}] "
            f"Epoch {epoch:02d}/{epochs} | "
            f"train_scaled_mse={train_loss:.6f} | "
            f"val_mse={val_metrics['mse']:.6f} | "
            f"val_mae={val_metrics['mae']:.6f} | "
            f"val_rmse={val_metrics['rmse']:.6f}"
        )

        if patience_left <= 0:
            print(f"Early stopping triggered for {dataset_name} | {model_name}")
            break

    total_seconds = time.time() - t0
    safe_json_dump(history, hist_path)

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    return {
        "history_path": str(hist_path),
        "ckpt_path": str(ckpt_path),
        "best_epoch": int(best_epoch),
        "best_val_mse": float(best_val),
        "train_seconds": float(total_seconds),
    }

In [3]:
# ============================================
# Cell 3: Full Phase A run across all 9 datasets
# ============================================

set_seed(GLOBAL_SEED)

all_results = []

for dataset_name, ds_cfg in DATASET_RUN_CONFIG.items():
    print("\n" + "=" * 130)
    print(f"RUNNING DATASET: {dataset_name}")
    print("=" * 130)

    bundle = load_prepared_dataset(dataset_name)

    seq_len = ds_cfg["seq_len"]
    pred_len = ds_cfg["pred_len"]
    seasonality = ds_cfg["seasonality"]
    batch_size = ds_cfg["batch_size"]
    epochs = ds_cfg["epochs"]

    input_dim = int(bundle["values_scaled"].shape[1])
    target_idx = int(bundle["target_idx"])
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    print(
        f"input_dim={input_dim}, target_idx={target_idx}, "
        f"seq_len={seq_len}, pred_len={pred_len}, seasonality={seasonality}, batch_size={batch_size}"
    )

    train_loader, val_loader, test_loader = make_loaders(
        bundle=bundle,
        seq_len=seq_len,
        pred_len=pred_len,
        batch_size=batch_size,
        num_workers=TRAIN_CFG["num_workers"],
    )

    # ----------------------------------------
    # 1) Naive baseline
    # ----------------------------------------
    print(f"\n[{dataset_name}] Running naive baseline...")
    val_metrics, val_preds, val_trues = evaluate_naive_baseline(val_loader)
    test_metrics, test_preds, test_trues = evaluate_naive_baseline(test_loader)

    pred_dir = PRED_ROOT / dataset_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        pred_dir / "naive_test_predictions.npz",
        preds=test_preds,
        trues=test_trues,
        seq_len=seq_len,
        pred_len=pred_len,
    )

    all_results.append({
        "dataset": dataset_name,
        "model": "naive",
        "family": "classical",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "seasonality": seasonality,
        "params": 0,
        "best_epoch": None,
        "train_seconds": 0.0,
        "val_mse": val_metrics["mse"],
        "val_mae": val_metrics["mae"],
        "val_rmse": val_metrics["rmse"],
        "test_mse": test_metrics["mse"],
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "checkpoint": None,
        "history": None,
        "prediction_file": str((pred_dir / "naive_test_predictions.npz").relative_to(ROOT)),
    })

    # ----------------------------------------
    # 2) Seasonal naive baseline
    # ----------------------------------------
    print(f"\n[{dataset_name}] Running seasonal naive baseline...")
    val_metrics, val_preds, val_trues = evaluate_seasonal_naive_baseline(val_loader, seasonality=seasonality)
    test_metrics, test_preds, test_trues = evaluate_seasonal_naive_baseline(test_loader, seasonality=seasonality)

    np.savez_compressed(
        pred_dir / "seasonal_naive_test_predictions.npz",
        preds=test_preds,
        trues=test_trues,
        seq_len=seq_len,
        pred_len=pred_len,
        seasonality=seasonality,
    )

    all_results.append({
        "dataset": dataset_name,
        "model": "seasonal_naive",
        "family": "classical",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "seasonality": seasonality,
        "params": 0,
        "best_epoch": None,
        "train_seconds": 0.0,
        "val_mse": val_metrics["mse"],
        "val_mae": val_metrics["mae"],
        "val_rmse": val_metrics["rmse"],
        "test_mse": test_metrics["mse"],
        "test_mae": test_metrics["mae"],
        "test_rmse": test_metrics["rmse"],
        "checkpoint": None,
        "history": None,
        "prediction_file": str((pred_dir / "seasonal_naive_test_predictions.npz").relative_to(ROOT)),
    })

    # ----------------------------------------
    # 3) Neural models
    # ----------------------------------------
    for model_name in ["lstm", "gru", "dlinear", "fixed_dwt_lstm"]:
        print(f"\n[{dataset_name}] Running model: {model_name}")
        set_seed(GLOBAL_SEED)

        model = build_model(
            model_name=model_name,
            input_dim=input_dim,
            seq_len=seq_len,
            pred_len=pred_len,
            target_idx=target_idx,
        ).to(DEVICE)

        params = count_parameters(model)
        print(f"Model {model_name} params: {params:,}")

        train_info = train_with_early_stopping(
            model_name=model_name,
            model=model,
            dataset_name=dataset_name,
            train_loader=train_loader,
            val_loader=val_loader,
            target_mean=target_mean,
            target_std=target_std,
            epochs=epochs,
        )

        val_metrics, _, _ = evaluate_neural_model(model, val_loader, target_mean, target_std)
        test_metrics, test_preds, test_trues = evaluate_neural_model(model, test_loader, target_mean, target_std)

        pred_file = pred_dir / f"{model_name}_test_predictions.npz"
        np.savez_compressed(
            pred_file,
            preds=test_preds,
            trues=test_trues,
            seq_len=seq_len,
            pred_len=pred_len,
        )

        family_name = "neural_non_wavelet"
        if model_name == "fixed_dwt_lstm":
            family_name = "wavelet"

        all_results.append({
            "dataset": dataset_name,
            "model": model_name,
            "family": family_name,
            "seq_len": seq_len,
            "pred_len": pred_len,
            "seasonality": seasonality,
            "params": int(params),
            "best_epoch": train_info["best_epoch"],
            "train_seconds": train_info["train_seconds"],
            "val_mse": val_metrics["mse"],
            "val_mae": val_metrics["mae"],
            "val_rmse": val_metrics["rmse"],
            "test_mse": test_metrics["mse"],
            "test_mae": test_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
            "checkpoint": str(Path(train_info["ckpt_path"]).relative_to(ROOT)),
            "history": str(Path(train_info["history_path"]).relative_to(ROOT)),
            "prediction_file": str(pred_file.relative_to(ROOT)),
        })

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    # Save per-dataset interim CSV
    interim_df = pd.DataFrame([r for r in all_results if r["dataset"] == dataset_name])
    interim_path = TABLES_ROOT / f"{dataset_name}_phaseA_metrics.csv"
    interim_df.to_csv(interim_path, index=False)
    print(f"\nSaved interim metrics: {interim_path}")

# ----------------------------------------
# Save full results
# ----------------------------------------
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(["dataset", "family", "model"]).reset_index(drop=True)

full_csv_path = TABLES_ROOT / "phaseA_all_datasets_metrics.csv"
results_df.to_csv(full_csv_path, index=False)

print("\n" + "=" * 130)
print("PHASE A COMPLETE")
print(f"Full metrics saved to: {full_csv_path}")
print("=" * 130)

display(results_df)


RUNNING DATASET: etth1
input_dim=7, target_idx=6, seq_len=96, pred_len=96, seasonality=24, batch_size=256

[etth1] Running naive baseline...

[etth1] Running seasonal naive baseline...

[etth1] Running model: lstm
Model lstm params: 294,368
[etth1 | lstm] Epoch 01/20 | train_scaled_mse=0.381785 | val_mse=16.201819 | val_mae=3.293811 | val_rmse=4.025148
[etth1 | lstm] Epoch 02/20 | train_scaled_mse=0.178554 | val_mse=20.425735 | val_mae=3.692391 | val_rmse=4.519484
[etth1 | lstm] Epoch 03/20 | train_scaled_mse=0.181170 | val_mse=23.409232 | val_mae=3.970632 | val_rmse=4.838309
[etth1 | lstm] Epoch 04/20 | train_scaled_mse=0.154257 | val_mse=32.891680 | val_mae=4.864084 | val_rmse=5.735127
[etth1 | lstm] Epoch 05/20 | train_scaled_mse=0.150339 | val_mse=29.931677 | val_mae=4.534953 | val_rmse=5.470985
[etth1 | lstm] Epoch 06/20 | train_scaled_mse=0.136379 | val_mse=30.696709 | val_mae=4.604247 | val_rmse=5.540461
Early stopping triggered for etth1 | lstm

[etth1] Running model: gru
Mode

,dataset,model,family,seq_len,pred_len,seasonality,params,best_epoch,train_seconds,val_mse,val_mae,val_rmse,test_mse,test_mae,test_rmse,checkpoint,history,prediction_file
0,electricity,naive,classical,168,96,24,0,NaN,0.000000,3.538033e+05,451.700746,594.813638,5.027207e+05,552.290455,709.028016,None,None,results/predictions/electricity/naive_test_pre...
1,electricity,seasonal_naive,classical,168,96,24,0,NaN,0.000000,1.743285e+05,297.186178,417.526660,2.405028e+05,356.983977,490.410830,None,None,results/predictions/electricity/seasonal_naive...
2,electricity,dlinear,neural_non_wavelet,168,96,24,32448,7.0,39.167233,5.969671e+04,163.575339,244.329101,7.355014e+04,191.765079,271.201295,results/checkpoints/electricity/dlinear.pt,results/histories/electricity/dlinear_history....,results/predictions/electricity/dlinear_test_p...
3,electricity,gru,neural_non_wavelet,168,96,24,268512,1.0,18.775522,1.106487e+05,240.066187,332.638936,1.558839e+05,292.559933,394.821400,results/checkpoints/electricity/gru.pt,results/histories/electricity/gru_history.json,results/predictions/electricity/gru_test_predi...
4,electricity,lstm,neural_non_wavelet,168,96,24,334560,1.0,18.549354,1.140757e+05,240.926217,337.751010,1.433671e+05,283.256661,378.638483,results/checkpoints/electricity/lstm.pt,results/histories/electricity/lstm_history.json,results/predictions/electricity/lstm_test_pred...
5,electricity,fixed_dwt_lstm,wavelet,168,96,24,416736,1.0,20.493988,1.166750e+05,243.417874,341.577250,1.523836e+05,287.094545,390.363431,results/checkpoints/electricity/fixed_dwt_lstm.pt,results/histories/electricity/fixed_dwt_lstm_h...,results/predictions/electricity/fixed_dwt_lstm...
6,etth1,naive,classical,96,96,24,0,NaN,0.000000,1.155812e+01,2.598826,3.399723,5.832596e+00,1.865423,2.415077,None,None,results/predictions/etth1/naive_test_predictio...
7,etth1,seasonal_naive,classical,96,96,24,0,NaN,0.000000,1.179727e+01,2.621478,3.434715,6.016950e+00,1.931772,2.452947,None,None,results/predictions/etth1/seasonal_naive_test_...
8,etth1,dlinear,neural_non_wavelet,96,96,24,18624,20.0,9.327195,8.209501e+00,2.233150,2.865223,6.053799e+00,1.817606,2.460447,results/checkpoints/etth1/dlinear.pt,results/histories/etth1/dlinear_history.json,results/predictions/etth1/dlinear_test_predict...
9,etth1,gru,neural_non_wavelet,96,96,24,228320,2.0,4.891402,1.612535e+01,3.259366,4.015638,3.423901e+01,5.397850,5.851411,results/checkpoints/etth1/gru.pt,results/histories/etth1/gru_history.json,results/predictions/etth1/gru_test_predictions...


In [4]:
# ============================================
# Cell 4: Summary views and ranking tables
# ============================================

results_path = ROOT / "results" / "tables" / "phaseA_all_datasets_metrics.csv"
results_df = pd.read_csv(results_path)

print("Full results file:", results_path)
display(results_df)

print("\nBest model per dataset by TEST RMSE")
best_rmse = (
    results_df.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
    .groupby("dataset", as_index=False)
    .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse", "seq_len", "pred_len"]]
)
display(best_rmse)

print("\nAverage test metrics across datasets by model")
avg_by_model = (
    results_df.groupby(["model", "family"], as_index=False)[["test_mse", "test_mae", "test_rmse"]]
    .mean()
    .sort_values("test_rmse")
)
display(avg_by_model)

best_rmse.to_csv(ROOT / "results" / "tables" / "phaseA_best_by_dataset.csv", index=False)
avg_by_model.to_csv(ROOT / "results" / "tables" / "phaseA_average_by_model.csv", index=False)

print("\nSaved:")
print(ROOT / "results" / "tables" / "phaseA_best_by_dataset.csv")
print(ROOT / "results" / "tables" / "phaseA_average_by_model.csv")

Full results file: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_all_datasets_metrics.csv


,dataset,model,family,seq_len,pred_len,seasonality,params,best_epoch,train_seconds,val_mse,val_mae,val_rmse,test_mse,test_mae,test_rmse,checkpoint,history,prediction_file
0,electricity,naive,classical,168,96,24,0,NaN,0.000000,3.538033e+05,451.700746,594.813638,5.027207e+05,552.290455,709.028016,NaN,NaN,results/predictions/electricity/naive_test_pre...
1,electricity,seasonal_naive,classical,168,96,24,0,NaN,0.000000,1.743285e+05,297.186178,417.526660,2.405028e+05,356.983977,490.410830,NaN,NaN,results/predictions/electricity/seasonal_naive...
2,electricity,dlinear,neural_non_wavelet,168,96,24,32448,7.0,38.603244,5.969671e+04,163.575339,244.329101,7.355014e+04,191.765079,271.201295,results/checkpoints/electricity/dlinear.pt,results/histories/electricity/dlinear_history....,results/predictions/electricity/dlinear_test_p...
3,electricity,gru,neural_non_wavelet,168,96,24,268512,1.0,19.142688,1.106487e+05,240.066187,332.638936,1.558839e+05,292.559933,394.821400,results/checkpoints/electricity/gru.pt,results/histories/electricity/gru_history.json,results/predictions/electricity/gru_test_predi...
4,electricity,lstm,neural_non_wavelet,168,96,24,334560,1.0,19.216703,1.140757e+05,240.926217,337.751010,1.433671e+05,283.256661,378.638483,results/checkpoints/electricity/lstm.pt,results/histories/electricity/lstm_history.json,results/predictions/electricity/lstm_test_pred...
5,electricity,fixed_dwt_lstm,wavelet,168,96,24,416736,1.0,19.577910,1.166750e+05,243.417874,341.577250,1.523836e+05,287.094545,390.363431,results/checkpoints/electricity/fixed_dwt_lstm.pt,results/histories/electricity/fixed_dwt_lstm_h...,results/predictions/electricity/fixed_dwt_lstm...
6,etth1,naive,classical,96,96,24,0,NaN,0.000000,1.155812e+01,2.598826,3.399723,5.832596e+00,1.865423,2.415077,NaN,NaN,results/predictions/etth1/naive_test_predictio...
7,etth1,seasonal_naive,classical,96,96,24,0,NaN,0.000000,1.179727e+01,2.621478,3.434715,6.016950e+00,1.931772,2.452947,NaN,NaN,results/predictions/etth1/seasonal_naive_test_...
8,etth1,dlinear,neural_non_wavelet,96,96,24,18624,20.0,9.009015,8.209501e+00,2.233150,2.865223,6.053799e+00,1.817606,2.460447,results/checkpoints/etth1/dlinear.pt,results/histories/etth1/dlinear_history.json,results/predictions/etth1/dlinear_test_predict...
9,etth1,gru,neural_non_wavelet,96,96,24,228320,2.0,4.386834,1.612535e+01,3.259366,4.015638,3.423901e+01,5.397850,5.851411,results/checkpoints/etth1/gru.pt,results/histories/etth1/gru_history.json,results/predictions/etth1/gru_test_predictions...



Best model per dataset by TEST RMSE


,dataset,model,family,test_mse,test_mae,test_rmse,seq_len,pred_len
0,electricity,dlinear,neural_non_wavelet,7.355014e+04,191.765079,271.201295,168,96
1,etth1,naive,classical,5.832596e+00,1.865423,2.415077,96,96
2,etth2,dlinear,neural_non_wavelet,1.748497e+01,3.192522,4.181504,96,96
3,ettm1,naive,classical,2.798778e+00,1.251871,1.672955,192,96
4,ettm2,dlinear,neural_non_wavelet,8.891993e+00,2.170217,2.981944,192,96
5,exchange,naive,classical,7.954944e-04,0.021018,0.028205,96,96
6,ili,dlinear,neural_non_wavelet,3.750925e+10,156273.240794,193673.051181,104,24
7,traffic,dlinear,neural_non_wavelet,4.471525e-05,0.003913,0.006687,168,96
8,weather,naive,classical,1.832375e+02,9.531391,13.536526,288,96



Average test metrics across datasets by model


,model,family,test_mse,test_mae,test_rmse
0,dlinear,neural_non_wavelet,4.167703e+09,17388.461999,21553.717208
5,seasonal_naive,classical,5.864791e+09,23480.961399,25584.860771
4,naive,classical,8.302916e+09,22640.964699,30455.402957
2,gru,neural_non_wavelet,2.220643e+10,42873.970185,49723.369586
3,lstm,neural_non_wavelet,2.260229e+10,43506.640620,50161.410436
1,fixed_dwt_lstm,wavelet,2.318145e+10,43607.838272,50801.587063



Saved:
/data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_best_by_dataset.csv
/data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_average_by_model.csv


In [1]:
from pathlib import Path
import csv
import time
import subprocess
import numpy as np

PROJECT_ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
TSLIB_ROOT = PROJECT_ROOT / "external" / "TSLib"
RESULTS_TABLES = PROJECT_ROOT / "results" / "tables"
RESULTS_LOGS = PROJECT_ROOT / "results" / "logs" / "tslib_modern"
CHECKPOINTS = PROJECT_ROOT / "results" / "tslib_checkpoints"

RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_LOGS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

OUT_CSV = RESULTS_TABLES / "phaseB_tslib_modern_metrics.csv"

MODELS = [
    "Autoformer",
    "FEDformer",
    "TimesNet",
    "iTransformer",
    "TimeMixer",
    "WPMixer",
]

# FFT-based models -> no AMP
NO_AMP_MODELS = {"Autoformer", "FEDformer", "TimesNet"}

DATASETS = {
    "etth1": {
        "data": "ETTh1",
        "root_path": str(TSLIB_ROOT / "dataset" / "ETT-small"),
        "data_path": "ETTh1.csv",
        "freq": "h",
        "enc_in": 7,
        "seq_len": 96,
        "pred_len": 96,
        "label_len": 48,
    },
    "etth2": {
        "data": "ETTh2",
        "root_path": str(TSLIB_ROOT / "dataset" / "ETT-small"),
        "data_path": "ETTh2.csv",
        "freq": "h",
        "enc_in": 7,
        "seq_len": 96,
        "pred_len": 96,
        "label_len": 48,
    },
    "ettm1": {
        "data": "ETTm1",
        "root_path": str(TSLIB_ROOT / "dataset" / "ETT-small"),
        "data_path": "ETTm1.csv",
        "freq": "t",
        "enc_in": 7,
        "seq_len": 192,
        "pred_len": 96,
        "label_len": 96,
    },
    "ettm2": {
        "data": "ETTm2",
        "root_path": str(TSLIB_ROOT / "dataset" / "ETT-small"),
        "data_path": "ETTm2.csv",
        "freq": "t",
        "enc_in": 7,
        "seq_len": 192,
        "pred_len": 96,
        "label_len": 96,
    },
    "weather": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "weather"),
        "data_path": "weather.csv",
        "freq": "10min",
        "enc_in": 21,
        "seq_len": 288,
        "pred_len": 96,
        "label_len": 144,
    },
    "electricity": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "electricity"),
        "data_path": "electricity.csv",
        "freq": "h",
        "enc_in": 321,
        "seq_len": 168,
        "pred_len": 96,
        "label_len": 84,
    },
    "traffic": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "traffic"),
        "data_path": "traffic.csv",
        "freq": "h",
        "enc_in": 862,
        "seq_len": 168,
        "pred_len": 96,
        "label_len": 84,
    },
    "exchange": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "exchange_rate"),
        "data_path": "exchange_rate.csv",
        "freq": "d",
        "enc_in": 8,
        "seq_len": 96,
        "pred_len": 96,
        "label_len": 48,
    },
    "ili": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "illness"),
        "data_path": "national_illness.csv",
        "freq": "w",
        "enc_in": 7,
        "seq_len": 104,
        "pred_len": 24,
        "label_len": 52,
    },
}

def gpu_mem_gb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            text=True
        ).strip().splitlines()[0]
        return int(out) / 1024.0
    except Exception:
        return 0.0

GPU_GB = gpu_mem_gb()

def pick_d_model(model: str, enc_in: int) -> int:
    if model in {"Autoformer", "FEDformer", "iTransformer"}:
        if enc_in > 400:
            return 128
        elif enc_in > 64:
            return 256
        else:
            return 512
    else:
        if enc_in > 400:
            return 64
        elif enc_in > 64:
            return 128
        else:
            return 192

def pick_batch_size(model: str, enc_in: int, gpu_gb: float) -> int:
    heavy = model in {"Autoformer", "FEDformer", "iTransformer"}
    if gpu_gb >= 70:
        if enc_in > 700:
            return 8 if heavy else 16
        elif enc_in > 300:
            return 16 if heavy else 24
        elif enc_in > 64:
            return 24 if heavy else 32
        else:
            return 32 if heavy else 48
    elif gpu_gb >= 40:
        if enc_in > 700:
            return 4 if heavy else 8
        elif enc_in > 300:
            return 8 if heavy else 12
        elif enc_in > 64:
            return 12 if heavy else 16
        else:
            return 16 if heavy else 24
    else:
        if enc_in > 700:
            return 2 if heavy else 4
        elif enc_in > 300:
            return 4 if heavy else 6
        elif enc_in > 64:
            return 6 if heavy else 8
        else:
            return 8 if heavy else 12

def append_row(row: dict):
    file_exists = OUT_CSV.exists()
    with open(OUT_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

def find_result_dir(model_id: str, model: str):
    results_root = TSLIB_ROOT / "results"
    candidates = sorted(
        [p for p in results_root.glob(f"*{model_id}_{model}_*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None

def run_one(dataset_name: str, model: str):
    cfg = DATASETS[dataset_name]
    enc_in = cfg["enc_in"]
    d_model = pick_d_model(model, enc_in)
    d_ff = d_model * 4
    batch_size = pick_batch_size(model, enc_in, GPU_GB)
    use_amp = model not in NO_AMP_MODELS

    model_id = f"{dataset_name}_{model}"
    log_file = RESULTS_LOGS / f"{dataset_name}__{model}.log"

    existing_dir = find_result_dir(model_id, model)
    if existing_dir is not None:
        metrics_path = existing_dir / "metrics.npy"
        pred_path = existing_dir / "pred.npy"
        true_path = existing_dir / "true.npy"
        if metrics_path.exists():
            metrics = np.load(metrics_path)
            mae, mse, rmse, mape, mspe = [float(x) for x in metrics.tolist()]
            append_row({
                "dataset": dataset_name,
                "model": model,
                "family": "modern_tslib",
                "seq_len": cfg["seq_len"],
                "label_len": cfg["label_len"],
                "pred_len": cfg["pred_len"],
                "enc_in": enc_in,
                "freq": cfg["freq"],
                "target": "OT",
                "features": "MS",
                "d_model": d_model,
                "d_ff": d_ff,
                "batch_size": batch_size,
                "train_epochs": 50,
                "patience": 10,
                "use_amp": use_amp,
                "mae": mae,
                "mse": mse,
                "rmse": rmse,
                "mape": mape,
                "mspe": mspe,
                "result_dir": str(existing_dir),
                "metrics_file": str(metrics_path),
                "pred_file": str(pred_path),
                "true_file": str(true_path),
                "log_file": str(log_file),
                "status": "skipped_existing",
            })
            return

    cmd = [
        "python", "-u", "run.py",
        "--task_name", "long_term_forecast",
        "--is_training", "1",
        "--root_path", cfg["root_path"],
        "--data_path", cfg["data_path"],
        "--model_id", model_id,
        "--model", model,
        "--data", cfg["data"],
        "--features", "MS",
        "--target", "OT",
        "--freq", cfg["freq"],
        "--seq_len", str(cfg["seq_len"]),
        "--label_len", str(cfg["label_len"]),
        "--pred_len", str(cfg["pred_len"]),
        "--enc_in", str(enc_in),
        "--dec_in", str(enc_in),
        "--c_out", str(enc_in),
        "--d_model", str(d_model),
        "--n_heads", "8",
        "--e_layers", "2",
        "--d_layers", "1",
        "--d_ff", str(d_ff),
        "--factor", "1",
        "--dropout", "0.1",
        "--train_epochs", "50",
        "--patience", "10",
        "--batch_size", str(batch_size),
        "--learning_rate", "0.0001",
        "--num_workers", "8",
        "--itr", "1",
        "--des", "phaseB_modern",
        "--checkpoints", str(CHECKPOINTS),
        "--gpu", "0",
    ]

    if use_amp:
        cmd.append("--use_amp")

    print("\n" + "=" * 100)
    print("RUN:", dataset_name, model, "| AMP:", use_amp, "| batch:", batch_size, "| d_model:", d_model)
    print("=" * 100)

    start = time.time()
    with open(log_file, "w") as lf:
        proc = subprocess.run(
            cmd,
            cwd=str(TSLIB_ROOT),
            stdout=lf,
            stderr=subprocess.STDOUT,
            text=True
        )
    wall_time = time.time() - start

    result_dir = find_result_dir(model_id, model)
    metrics_path = result_dir / "metrics.npy" if result_dir else None
    pred_path = result_dir / "pred.npy" if result_dir else None
    true_path = result_dir / "true.npy" if result_dir else None

    if proc.returncode != 0:
        append_row({
            "dataset": dataset_name,
            "model": model,
            "family": "modern_tslib",
            "seq_len": cfg["seq_len"],
            "label_len": cfg["label_len"],
            "pred_len": cfg["pred_len"],
            "enc_in": enc_in,
            "freq": cfg["freq"],
            "target": "OT",
            "features": "MS",
            "d_model": d_model,
            "d_ff": d_ff,
            "batch_size": batch_size,
            "train_epochs": 50,
            "patience": 10,
            "use_amp": use_amp,
            "mae": None,
            "mse": None,
            "rmse": None,
            "mape": None,
            "mspe": None,
            "result_dir": str(result_dir) if result_dir else None,
            "metrics_file": str(metrics_path) if metrics_path else None,
            "pred_file": str(pred_path) if pred_path else None,
            "true_file": str(true_path) if true_path else None,
            "log_file": str(log_file),
            "status": f"failed_returncode_{proc.returncode}",
        })
        return

    if result_dir is None or metrics_path is None or not metrics_path.exists():
        append_row({
            "dataset": dataset_name,
            "model": model,
            "family": "modern_tslib",
            "seq_len": cfg["seq_len"],
            "label_len": cfg["label_len"],
            "pred_len": cfg["pred_len"],
            "enc_in": enc_in,
            "freq": cfg["freq"],
            "target": "OT",
            "features": "MS",
            "d_model": d_model,
            "d_ff": d_ff,
            "batch_size": batch_size,
            "train_epochs": 50,
            "patience": 10,
            "use_amp": use_amp,
            "mae": None,
            "mse": None,
            "rmse": None,
            "mape": None,
            "mspe": None,
            "result_dir": str(result_dir) if result_dir else None,
            "metrics_file": str(metrics_path) if metrics_path else None,
            "pred_file": str(pred_path) if pred_path else None,
            "true_file": str(true_path) if true_path else None,
            "log_file": str(log_file),
            "status": "finished_but_metrics_missing",
        })
        return

    metrics = np.load(metrics_path)
    mae, mse, rmse, mape, mspe = [float(x) for x in metrics.tolist()]

    append_row({
        "dataset": dataset_name,
        "model": model,
        "family": "modern_tslib",
        "seq_len": cfg["seq_len"],
        "label_len": cfg["label_len"],
        "pred_len": cfg["pred_len"],
        "enc_in": enc_in,
        "freq": cfg["freq"],
        "target": "OT",
        "features": "MS",
        "d_model": d_model,
        "d_ff": d_ff,
        "batch_size": batch_size,
        "train_epochs": 50,
        "patience": 10,
        "use_amp": use_amp,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "mape": mape,
        "mspe": mspe,
        "result_dir": str(result_dir),
        "metrics_file": str(metrics_path),
        "pred_file": str(pred_path),
        "true_file": str(true_path),
        "log_file": str(log_file),
        "status": f"ok_walltime_sec_{round(wall_time, 2)}",
    })

def main():
    print("Detected GPU memory (GB):", GPU_GB)
    print("Output CSV:", OUT_CSV)
    for dataset_name in DATASETS:
        for model in MODELS:
            run_one(dataset_name, model)
    print("Saved:", OUT_CSV)

if __name__ == "__main__":
    main()

Detected GPU memory (GB): 93.583984375
Output CSV: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseB_tslib_modern_metrics.csv

RUN: etth1 TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: etth2 TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: ettm1 TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: ettm2 TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: weather Autoformer | AMP: False | batch: 32 | d_model: 512

RUN: weather FEDformer | AMP: False | batch: 32 | d_model: 512

RUN: weather TimesNet | AMP: False | batch: 48 | d_model: 192

RUN: weather TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: electricity TimeMixer | AMP: True | batch: 24 | d_model: 128

RUN: traffic TimeMixer | AMP: True | batch: 16 | d_model: 64

RUN: exchange TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: ili TimeMixer | AMP: True | batch: 48 | d_model: 192
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseB_tslib_modern_metrics.csv


In [3]:
%%bash
set -e

# If needed, adjust this env name
source ~/miniconda3/etc/profile.d/conda.sh 2>/dev/null || true
conda activate myenv 2>/dev/null || true

cd /data/Sajjan_Singh/spml/wavelet_seq_project

mkdir -p scripts results/logs/tslib_modern results/tables results/tslib_checkpoints

# Clean only the broken Phase B tracking files
rm -f results/tables/phaseB_tslib_modern_metrics.csv
rm -f results/tables/phaseB_best_by_dataset.csv
rm -f results/logs/phaseB_tslib_modern_full.log
find results/logs/tslib_modern -type f -delete || true

# Write patched runner
cat > scripts/run_tslib_modern.py << 'PY'
from pathlib import Path
import csv
import time
import subprocess
import numpy as np

PROJECT_ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
TSLIB_ROOT = PROJECT_ROOT / "external" / "TSLib"
RESULTS_TABLES = PROJECT_ROOT / "results" / "tables"
RESULTS_LOGS = PROJECT_ROOT / "results" / "logs" / "tslib_modern"
CHECKPOINTS = PROJECT_ROOT / "results" / "tslib_checkpoints"

RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_LOGS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

OUT_CSV = RESULTS_TABLES / "phaseB_tslib_modern_metrics.csv"

MODELS = [
    "Autoformer",
    "FEDformer",
    "TimesNet",
    "iTransformer",
    "TimeMixer",
    "WPMixer",
]

NO_AMP_MODELS = {"Autoformer", "FEDformer", "TimesNet"}

DATASETS = {
    "etth1": {
        "data": "ETTh1",
        "root_path": str(TSLIB_ROOT / "dataset" / "ETT-small"),
        "data_path": "ETTh1.csv",
        "freq": "h",
        "enc_in": 7,
        "seq_len": 96,
        "pred_len": 96,
        "label_len": 48,
    },
    "etth2": {
        "data": "ETTh2",
        "root_path": str(TSLIB_ROOT / "dataset" / "ETT-small"),
        "data_path": "ETTh2.csv",
        "freq": "h",
        "enc_in": 7,
        "seq_len": 96,
        "pred_len": 96,
        "label_len": 48,
    },
    "ettm1": {
        "data": "ETTm1",
        "root_path": str(TSLIB_ROOT / "dataset" / "ETT-small"),
        "data_path": "ETTm1.csv",
        "freq": "t",
        "enc_in": 7,
        "seq_len": 192,
        "pred_len": 96,
        "label_len": 96,
    },
    "ettm2": {
        "data": "ETTm2",
        "root_path": str(TSLIB_ROOT / "dataset" / "ETT-small"),
        "data_path": "ETTm2.csv",
        "freq": "t",
        "enc_in": 7,
        "seq_len": 192,
        "pred_len": 96,
        "label_len": 96,
    },
    "weather": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "weather"),
        "data_path": "weather.csv",
        "freq": "10min",
        "enc_in": 21,
        "seq_len": 288,
        "pred_len": 96,
        "label_len": 144,
    },
    "electricity": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "electricity"),
        "data_path": "electricity.csv",
        "freq": "h",
        "enc_in": 321,
        "seq_len": 168,
        "pred_len": 96,
        "label_len": 84,
    },
    "traffic": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "traffic"),
        "data_path": "traffic.csv",
        "freq": "h",
        "enc_in": 862,
        "seq_len": 168,
        "pred_len": 96,
        "label_len": 84,
    },
    "exchange": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "exchange_rate"),
        "data_path": "exchange_rate.csv",
        "freq": "d",
        "enc_in": 8,
        "seq_len": 96,
        "pred_len": 96,
        "label_len": 48,
    },
    "ili": {
        "data": "custom",
        "root_path": str(TSLIB_ROOT / "dataset" / "illness"),
        "data_path": "national_illness.csv",
        "freq": "w",
        "enc_in": 7,
        "seq_len": 104,
        "pred_len": 24,
        "label_len": 52,
    },
}

def gpu_mem_gb():
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
            text=True
        ).strip().splitlines()[0]
        return int(out) / 1024.0
    except Exception:
        return 0.0

GPU_GB = gpu_mem_gb()

def pick_d_model(model: str, enc_in: int) -> int:
    if model in {"Autoformer", "FEDformer", "iTransformer"}:
        if enc_in > 400:
            return 128
        elif enc_in > 64:
            return 256
        else:
            return 512
    else:
        if enc_in > 400:
            return 64
        elif enc_in > 64:
            return 128
        else:
            return 192

def pick_batch_size(model: str, enc_in: int, gpu_gb: float) -> int:
    heavy = model in {"Autoformer", "FEDformer", "iTransformer"}
    if gpu_gb >= 70:
        if enc_in > 700:
            return 8 if heavy else 16
        elif enc_in > 300:
            return 16 if heavy else 24
        elif enc_in > 64:
            return 24 if heavy else 32
        else:
            return 32 if heavy else 48
    elif gpu_gb >= 40:
        if enc_in > 700:
            return 4 if heavy else 8
        elif enc_in > 300:
            return 8 if heavy else 12
        elif enc_in > 64:
            return 12 if heavy else 16
        else:
            return 16 if heavy else 24
    else:
        if enc_in > 700:
            return 2 if heavy else 4
        elif enc_in > 300:
            return 4 if heavy else 6
        elif enc_in > 64:
            return 6 if heavy else 8
        else:
            return 8 if heavy else 12

def append_row(row: dict):
    file_exists = OUT_CSV.exists()
    with open(OUT_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

def find_result_dir(model_id: str, model: str):
    results_root = TSLIB_ROOT / "results"
    candidates = sorted(
        [p for p in results_root.glob(f"*{model_id}_{model}_*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None

def run_one(dataset_name: str, model: str):
    cfg = DATASETS[dataset_name]
    enc_in = cfg["enc_in"]
    d_model = pick_d_model(model, enc_in)
    d_ff = d_model * 4
    batch_size = pick_batch_size(model, enc_in, GPU_GB)
    use_amp = model not in NO_AMP_MODELS

    model_id = f"{dataset_name}_{model}"
    log_file = RESULTS_LOGS / f"{dataset_name}__{model}.log"

    existing_dir = find_result_dir(model_id, model)
    if existing_dir is not None:
        metrics_path = existing_dir / "metrics.npy"
        pred_path = existing_dir / "pred.npy"
        true_path = existing_dir / "true.npy"
        if metrics_path.exists():
            metrics = np.load(metrics_path)
            mae, mse, rmse, mape, mspe = [float(x) for x in metrics.tolist()]
            append_row({
                "dataset": dataset_name,
                "model": model,
                "family": "modern_tslib",
                "seq_len": cfg["seq_len"],
                "label_len": cfg["label_len"],
                "pred_len": cfg["pred_len"],
                "enc_in": enc_in,
                "freq": cfg["freq"],
                "target": "OT",
                "features": "MS",
                "d_model": d_model,
                "d_ff": d_ff,
                "batch_size": batch_size,
                "train_epochs": 50,
                "patience": 10,
                "use_amp": use_amp,
                "mae": mae,
                "mse": mse,
                "rmse": rmse,
                "mape": mape,
                "mspe": mspe,
                "result_dir": str(existing_dir),
                "metrics_file": str(metrics_path),
                "pred_file": str(pred_path),
                "true_file": str(true_path),
                "log_file": str(log_file),
                "status": "skipped_existing",
            })
            return

    cmd = [
        "python", "-u", "run.py",
        "--task_name", "long_term_forecast",
        "--is_training", "1",
        "--root_path", cfg["root_path"],
        "--data_path", cfg["data_path"],
        "--model_id", model_id,
        "--model", model,
        "--data", cfg["data"],
        "--features", "MS",
        "--target", "OT",
        "--freq", cfg["freq"],
        "--seq_len", str(cfg["seq_len"]),
        "--label_len", str(cfg["label_len"]),
        "--pred_len", str(cfg["pred_len"]),
        "--enc_in", str(enc_in),
        "--dec_in", str(enc_in),
        "--c_out", str(enc_in),
        "--d_model", str(d_model),
        "--n_heads", "8",
        "--e_layers", "2",
        "--d_layers", "1",
        "--d_ff", str(d_ff),
        "--factor", "1",
        "--dropout", "0.1",
        "--train_epochs", "50",
        "--patience", "10",
        "--batch_size", str(batch_size),
        "--learning_rate", "0.0001",
        "--num_workers", "8",
        "--itr", "1",
        "--des", "phaseB_modern",
        "--checkpoints", str(CHECKPOINTS),
        "--gpu", "0",
    ]

    if use_amp:
        cmd.append("--use_amp")

    print("\n" + "=" * 100)
    print("RUN:", dataset_name, model, "| AMP:", use_amp, "| batch:", batch_size, "| d_model:", d_model)
    print("=" * 100)

    start = time.time()
    with open(log_file, "w") as lf:
        proc = subprocess.run(
            cmd,
            cwd=str(TSLIB_ROOT),
            stdout=lf,
            stderr=subprocess.STDOUT,
            text=True
        )
    wall_time = time.time() - start

    result_dir = find_result_dir(model_id, model)
    metrics_path = result_dir / "metrics.npy" if result_dir else None
    pred_path = result_dir / "pred.npy" if result_dir else None
    true_path = result_dir / "true.npy" if result_dir else None

    if proc.returncode != 0:
        append_row({
            "dataset": dataset_name,
            "model": model,
            "family": "modern_tslib",
            "seq_len": cfg["seq_len"],
            "label_len": cfg["label_len"],
            "pred_len": cfg["pred_len"],
            "enc_in": enc_in,
            "freq": cfg["freq"],
            "target": "OT",
            "features": "MS",
            "d_model": d_model,
            "d_ff": d_ff,
            "batch_size": batch_size,
            "train_epochs": 50,
            "patience": 10,
            "use_amp": use_amp,
            "mae": None,
            "mse": None,
            "rmse": None,
            "mape": None,
            "mspe": None,
            "result_dir": str(result_dir) if result_dir else None,
            "metrics_file": str(metrics_path) if metrics_path else None,
            "pred_file": str(pred_path) if pred_path else None,
            "true_file": str(true_path) if true_path else None,
            "log_file": str(log_file),
            "status": f"failed_returncode_{proc.returncode}",
        })
        return

    if result_dir is None or metrics_path is None or not metrics_path.exists():
        append_row({
            "dataset": dataset_name,
            "model": model,
            "family": "modern_tslib",
            "seq_len": cfg["seq_len"],
            "label_len": cfg["label_len"],
            "pred_len": cfg["pred_len"],
            "enc_in": enc_in,
            "freq": cfg["freq"],
            "target": "OT",
            "features": "MS",
            "d_model": d_model,
            "d_ff": d_ff,
            "batch_size": batch_size,
            "train_epochs": 50,
            "patience": 10,
            "use_amp": use_amp,
            "mae": None,
            "mse": None,
            "rmse": None,
            "mape": None,
            "mspe": None,
            "result_dir": str(result_dir) if result_dir else None,
            "metrics_file": str(metrics_path) if metrics_path else None,
            "pred_file": str(pred_path) if pred_path else None,
            "true_file": str(true_path) if true_path else None,
            "log_file": str(log_file),
            "status": "finished_but_metrics_missing",
        })
        return

    metrics = np.load(metrics_path)
    mae, mse, rmse, mape, mspe = [float(x) for x in metrics.tolist()]

    append_row({
        "dataset": dataset_name,
        "model": model,
        "family": "modern_tslib",
        "seq_len": cfg["seq_len"],
        "label_len": cfg["label_len"],
        "pred_len": cfg["pred_len"],
        "enc_in": enc_in,
        "freq": cfg["freq"],
        "target": "OT",
        "features": "MS",
        "d_model": d_model,
        "d_ff": d_ff,
        "batch_size": batch_size,
        "train_epochs": 50,
        "patience": 10,
        "use_amp": use_amp,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "mape": mape,
        "mspe": mspe,
        "result_dir": str(result_dir),
        "metrics_file": str(metrics_path),
        "pred_file": str(pred_path),
        "true_file": str(true_path),
        "log_file": str(log_file),
        "status": f"ok_walltime_sec_{round(wall_time, 2)}",
    })

def main():
    print("Detected GPU memory (GB):", GPU_GB)
    print("Output CSV:", OUT_CSV)
    for dataset_name in DATASETS:
        for model in MODELS:
            run_one(dataset_name, model)
    print("Saved:", OUT_CSV)

if __name__ == "__main__":
    main()
PY

echo "Patched runner written."

Patched runner written.


In [4]:
%%bash
set -e
source ~/miniconda3/etc/profile.d/conda.sh 2>/dev/null || true
conda activate myenv 2>/dev/null || true

cd /data/Sajjan_Singh/spml/wavelet_seq_project
python scripts/run_tslib_modern.py 2>&1 | tee results/logs/phaseB_tslib_modern_full.log

Detected GPU memory (GB): 93.583984375
Output CSV: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseB_tslib_modern_metrics.csv

RUN: etth1 TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: etth2 TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: ettm1 TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: ettm2 TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: weather Autoformer | AMP: False | batch: 32 | d_model: 512

RUN: weather FEDformer | AMP: False | batch: 32 | d_model: 512

RUN: weather TimesNet | AMP: False | batch: 48 | d_model: 192

RUN: weather TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: electricity TimeMixer | AMP: True | batch: 24 | d_model: 128

RUN: traffic TimeMixer | AMP: True | batch: 16 | d_model: 64

RUN: exchange TimeMixer | AMP: True | batch: 48 | d_model: 192

RUN: ili TimeMixer | AMP: True | batch: 48 | d_model: 192
Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseB_tslib_modern_metrics.csv


In [5]:
from pathlib import Path
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")

# ------------------------------------------------------------
# Verify Phase B
# ------------------------------------------------------------
phase_b = ROOT / "results" / "tables" / "phaseB_tslib_modern_metrics.csv"
print("Checking:", phase_b)

if not phase_b.exists():
    raise FileNotFoundError(f"Missing modern-stage metrics file: {phase_b}")

df_b = pd.read_csv(phase_b)

print("\nPhase B total rows:", len(df_b))
print("Expected rows:", 54)
print("Datasets:", sorted(df_b["dataset"].dropna().unique().tolist()))
print("Models:", sorted(df_b["model"].dropna().unique().tolist()))

print("\nPhase B status counts:")
print(df_b["status"].astype(str).value_counts(dropna=False).to_string())

ok_b = df_b[df_b["status"].astype(str).str.startswith(("ok_", "skipped_existing"))].copy()
print("\nCompleted/usable modern runs:", len(ok_b))

if len(ok_b) > 0:
    best_b = (
        ok_b.sort_values(["dataset", "rmse", "mae", "mse"])
            .groupby("dataset", as_index=False)
            .first()
    )
    show_cols = [c for c in ["dataset", "model", "mse", "mae", "rmse", "use_amp", "batch_size", "d_model", "status"] if c in best_b.columns]
    print("\nBest modern model per dataset by rmse:")
    print(best_b[show_cols].to_string(index=False))

    phase_b_best = ROOT / "results" / "tables" / "phaseB_best_by_dataset.csv"
    best_b.to_csv(phase_b_best, index=False)
    print("\nSaved:", phase_b_best)
else:
    print("\nNo usable modern runs found yet. Check logs in results/logs/tslib_modern/")

# ------------------------------------------------------------
# Merge Phase A + Phase B
# ------------------------------------------------------------
phase_a = ROOT / "results" / "tables" / "phaseA_all_datasets_metrics.csv"
if not phase_a.exists():
    raise FileNotFoundError(f"Missing Phase A metrics file: {phase_a}")

df_a = pd.read_csv(phase_a)
df_b_ok = ok_b.copy()

rename_map = {
    "mae": "test_mae",
    "mse": "test_mse",
    "rmse": "test_rmse",
}
df_b_ok = df_b_ok.rename(columns=rename_map)

common_cols = [
    "dataset", "model", "family",
    "seq_len", "pred_len",
    "params", "best_epoch", "train_seconds",
    "val_mse", "val_mae", "val_rmse",
    "test_mse", "test_mae", "test_rmse",
    "checkpoint", "history", "prediction_file",
]

for col in common_cols:
    if col not in df_a.columns:
        df_a[col] = None
    if col not in df_b_ok.columns:
        df_b_ok[col] = None

master = pd.concat([df_a[common_cols], df_b_ok[common_cols]], axis=0, ignore_index=True)
master = master.sort_values(["dataset", "family", "model"]).reset_index(drop=True)

master_path = ROOT / "results" / "tables" / "master_all_models_metrics.csv"
master.to_csv(master_path, index=False)

print("\nSaved master file:", master_path)
print("Master rows:", len(master))
print("Datasets:", sorted(master["dataset"].dropna().unique().tolist()))
print("Models:", sorted(master["model"].dropna().unique().tolist()))

best_master = (
    master.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
          .groupby("dataset", as_index=False)
          .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
)

print("\nBest overall model per dataset by test_rmse:")
print(best_master.to_string(index=False))

master_best_path = ROOT / "results" / "tables" / "master_best_by_dataset.csv"
best_master.to_csv(master_best_path, index=False)
print("\nSaved:", master_best_path)

Checking: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseB_tslib_modern_metrics.csv

Phase B total rows: 54
Expected rows: 54
Datasets: ['electricity', 'etth1', 'etth2', 'ettm1', 'ettm2', 'exchange', 'ili', 'traffic', 'weather']
Models: ['Autoformer', 'FEDformer', 'TimeMixer', 'TimesNet', 'WPMixer', 'iTransformer']

Phase B status counts:
status
skipped_existing       42
failed_returncode_1    12

Completed/usable modern runs: 42

Best modern model per dataset by rmse:
    dataset        model      mse      mae     rmse  use_amp  batch_size  d_model           status
electricity      WPMixer 0.218807 0.328796 0.467769     True          24      128 skipped_existing
      etth1      WPMixer 0.055708 0.178231 0.236025     True          48      192 skipped_existing
      etth2      WPMixer 0.132968 0.280923 0.364647     True          48      192 skipped_existing
      ettm1      WPMixer 0.027799 0.125130 0.166729     True          48      192 skipped_existing
      ettm2   

/tmp/ipykernel_1197340/2472037828.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master = pd.concat([df_a[common_cols], df_b_ok[common_cols]], axis=0, ignore_index=True)


In [6]:
from pathlib import Path
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
phase_b = ROOT / "results" / "tables" / "phaseB_tslib_modern_metrics.csv"

df = pd.read_csv(phase_b)

failed = df[df["status"].astype(str).str.startswith("failed_")].copy()

print("Total failed runs:", len(failed))
print("\nFailed runs summary:")
print(failed[["dataset", "model", "status", "log_file"]].to_string(index=False))

failed_out = ROOT / "results" / "tables" / "phaseB_failed_runs.csv"
failed.to_csv(failed_out, index=False)
print("\nSaved failed-run table to:", failed_out)

print("\nCount of failures by model:")
print(failed.groupby("model").size().sort_values(ascending=False).to_string())

print("\nCount of failures by dataset:")
print(failed.groupby("dataset").size().sort_values(ascending=False).to_string())

Total failed runs: 12

Failed runs summary:
    dataset      model              status                                                                                         log_file
      etth1  TimeMixer failed_returncode_1       /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/tslib_modern/etth1__TimeMixer.log
      etth2  TimeMixer failed_returncode_1       /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/tslib_modern/etth2__TimeMixer.log
      ettm1  TimeMixer failed_returncode_1       /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/tslib_modern/ettm1__TimeMixer.log
      ettm2  TimeMixer failed_returncode_1       /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/tslib_modern/ettm2__TimeMixer.log
    weather Autoformer failed_returncode_1    /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/tslib_modern/weather__Autoformer.log
    weather  FEDformer failed_returncode_1     /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/tslib_modern

In [7]:
from pathlib import Path
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
phase_b = ROOT / "results" / "tables" / "phaseB_tslib_modern_metrics.csv"

df = pd.read_csv(phase_b)
failed = df[df["status"].astype(str).str.startswith("failed_")].copy()

for _, row in failed.iterrows():
    log_path = Path(row["log_file"])
    print("\n" + "=" * 120)
    print(f"FAILED: dataset={row['dataset']} | model={row['model']}")
    print(f"LOG: {log_path}")
    print("=" * 120)

    if log_path.exists():
        with open(log_path, "r", errors="ignore") as f:
            lines = f.readlines()
        tail = "".join(lines[-80:])
        print(tail)
    else:
        print("Log file not found.")


FAILED: dataset=etth1 | model=TimeMixer
LOG: /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/tslib_modern/etth1__TimeMixer.log
Using GPU
Args in experiment:
Basic Config
  Task Name:          long_term_forecast  Is Training:        1                   
  Model ID:           etth1_TimeMixer     Model:              TimeMixer           

Data Loader
  Data:               ETTh1               Root Path:          /data/Sajjan_Singh/spml/wavelet_seq_project/external/TSLib/dataset/ETT-small
  Data Path:          ETTh1.csv           Features:           MS                  
  Target:             OT                  Freq:               h                   
  Checkpoints:        /data/Sajjan_Singh/spml/wavelet_seq_project/results/tslib_checkpoints

Forecasting Task
  Seq Len:            96                  Label Len:          48                  
  Pred Len:           96                  Seasonal Patterns:  Monthly             
  Inverse:            0                   

Model Parameters

In [8]:
from pathlib import Path
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
master_path = ROOT / "results" / "tables" / "master_all_models_metrics.csv"

df = pd.read_csv(master_path)

print("Master rows:", len(df))
print("Datasets:", sorted(df["dataset"].dropna().unique().tolist()))
print("Models:", sorted(df["model"].dropna().unique().tolist()))

best = (
    df.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
      .groupby("dataset", as_index=False)
      .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
)

print("\nBest overall model per dataset:")
print(best.to_string(index=False))

Master rows: 96
Datasets: ['electricity', 'etth1', 'etth2', 'ettm1', 'ettm2', 'exchange', 'ili', 'traffic', 'weather']
Models: ['Autoformer', 'FEDformer', 'TimesNet', 'WPMixer', 'dlinear', 'fixed_dwt_lstm', 'gru', 'iTransformer', 'lstm', 'naive', 'seasonal_naive']

Best overall model per dataset:
    dataset        model             family  test_mse  test_mae  test_rmse
electricity      WPMixer       modern_tslib  0.218807  0.328796   0.467769
      etth1      WPMixer       modern_tslib  0.055708  0.178231   0.236025
      etth2      WPMixer       modern_tslib  0.132968  0.280923   0.364647
      ettm1      WPMixer       modern_tslib  0.027799  0.125130   0.166729
      ettm2      WPMixer       modern_tslib  0.063345  0.182120   0.251685
   exchange        naive          classical  0.000795  0.021018   0.028205
        ili iTransformer       modern_tslib  0.628020  0.549220   0.792477
    traffic      dlinear neural_non_wavelet  0.000045  0.003913   0.006687
    weather      WPMixer   

In [9]:
from pathlib import Path
import subprocess
import time
import numpy as np
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
TSLIB = ROOT / "external" / "TSLib"
PHASE_B = ROOT / "results" / "tables" / "phaseB_tslib_modern_metrics.csv"
LOG_DIR = ROOT / "results" / "logs" / "tslib_modern"
CKPT_DIR = ROOT / "results" / "tslib_checkpoints"

LOG_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PHASE_B)

def find_result_dir(model_id: str, model: str):
    results_root = TSLIB / "results"
    candidates = sorted(
        [p for p in results_root.glob(f"*{model_id}_{model}_*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None

def upsert_row(df, row):
    mask = (df["dataset"] == row["dataset"]) & (df["model"] == row["model"])
    if mask.any():
        for k, v in row.items():
            if k not in df.columns:
                df[k] = None
            df.loc[mask, k] = v
    else:
        for k in row.keys():
            if k not in df.columns:
                df[k] = None
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return df

# Weather reruns with corrected freq='t'
weather_runs = [
    {
        "model": "Autoformer",
        "d_model": 512,
        "d_ff": 2048,
        "batch_size": 32,
        "use_amp": False,
    },
    {
        "model": "FEDformer",
        "d_model": 512,
        "d_ff": 2048,
        "batch_size": 32,
        "use_amp": False,
    },
    {
        "model": "TimesNet",
        "d_model": 192,
        "d_ff": 768,
        "batch_size": 48,
        "use_amp": False,
    },
]

for cfg in weather_runs:
    model = cfg["model"]
    model_id = f"weather_{model}"
    log_file = LOG_DIR / f"weather__{model}.log"

    cmd = [
        "python", "-u", "run.py",
        "--task_name", "long_term_forecast",
        "--is_training", "1",
        "--root_path", str(TSLIB / "dataset" / "weather"),
        "--data_path", "weather.csv",
        "--model_id", model_id,
        "--model", model,
        "--data", "custom",
        "--features", "MS",
        "--target", "OT",
        "--freq", "t",   # fixed from 10min -> t
        "--seq_len", "288",
        "--label_len", "144",
        "--pred_len", "96",
        "--enc_in", "21",
        "--dec_in", "21",
        "--c_out", "21",
        "--d_model", str(cfg["d_model"]),
        "--n_heads", "8",
        "--e_layers", "2",
        "--d_layers", "1",
        "--d_ff", str(cfg["d_ff"]),
        "--factor", "1",
        "--dropout", "0.1",
        "--train_epochs", "50",
        "--patience", "10",
        "--batch_size", str(cfg["batch_size"]),
        "--learning_rate", "0.0001",
        "--num_workers", "8",
        "--itr", "1",
        "--des", "phaseB_modern_weatherfix",
        "--checkpoints", str(CKPT_DIR),
        "--gpu", "0",
    ]

    if cfg["use_amp"]:
        cmd.append("--use_amp")

    print("\n" + "=" * 100)
    print("RUNNING:", model)
    print("CMD:", " ".join(cmd))
    print("=" * 100)

    start = time.time()
    with open(log_file, "w") as lf:
        proc = subprocess.run(cmd, cwd=str(TSLIB), stdout=lf, stderr=subprocess.STDOUT, text=True)
    wall = round(time.time() - start, 2)

    result_dir = find_result_dir(model_id, model)
    metrics_path = result_dir / "metrics.npy" if result_dir else None
    pred_path = result_dir / "pred.npy" if result_dir else None
    true_path = result_dir / "true.npy" if result_dir else None

    if proc.returncode == 0 and metrics_path is not None and metrics_path.exists():
        metrics = np.load(metrics_path)
        mae, mse, rmse, mape, mspe = [float(x) for x in metrics.tolist()]
        row = {
            "dataset": "weather",
            "model": model,
            "family": "modern_tslib",
            "seq_len": 288,
            "label_len": 144,
            "pred_len": 96,
            "enc_in": 21,
            "freq": "t",
            "target": "OT",
            "features": "MS",
            "d_model": cfg["d_model"],
            "d_ff": cfg["d_ff"],
            "batch_size": cfg["batch_size"],
            "train_epochs": 50,
            "patience": 10,
            "use_amp": cfg["use_amp"],
            "mae": mae,
            "mse": mse,
            "rmse": rmse,
            "mape": mape,
            "mspe": mspe,
            "result_dir": str(result_dir),
            "metrics_file": str(metrics_path),
            "pred_file": str(pred_path) if pred_path else None,
            "true_file": str(true_path) if true_path else None,
            "log_file": str(log_file),
            "status": f"ok_walltime_sec_{wall}",
        }
        print(f"{model}: SUCCESS | mse={mse:.6f} mae={mae:.6f} rmse={rmse:.6f}")
    else:
        row = {
            "dataset": "weather",
            "model": model,
            "status": f"failed_returncode_{proc.returncode}",
            "log_file": str(log_file),
        }
        print(f"{model}: FAILED | see {log_file}")

    df = upsert_row(df, row)

# Mark all failed TimeMixer runs as excluded under current MS protocol
mask_tm = (df["model"] == "TimeMixer") & (df["status"].astype(str).str.startswith("failed_"))
df.loc[mask_tm, "status"] = "excluded_incompatible_MS_protocol"

# Save cleaned Phase B CSV
df = df.sort_values(["dataset", "model"]).reset_index(drop=True)
df.to_csv(PHASE_B, index=False)

print("\nSaved cleaned Phase B file:", PHASE_B)
print("\nUpdated status counts:")
print(df["status"].astype(str).value_counts(dropna=False).to_string())


RUNNING: Autoformer
CMD: python -u run.py --task_name long_term_forecast --is_training 1 --root_path /data/Sajjan_Singh/spml/wavelet_seq_project/external/TSLib/dataset/weather --data_path weather.csv --model_id weather_Autoformer --model Autoformer --data custom --features MS --target OT --freq t --seq_len 288 --label_len 144 --pred_len 96 --enc_in 21 --dec_in 21 --c_out 21 --d_model 512 --n_heads 8 --e_layers 2 --d_layers 1 --d_ff 2048 --factor 1 --dropout 0.1 --train_epochs 50 --patience 10 --batch_size 32 --learning_rate 0.0001 --num_workers 8 --itr 1 --des phaseB_modern_weatherfix --checkpoints /data/Sajjan_Singh/spml/wavelet_seq_project/results/tslib_checkpoints --gpu 0
Autoformer: SUCCESS | mse=0.011472 mae=0.077626 rmse=0.107108

RUNNING: FEDformer
CMD: python -u run.py --task_name long_term_forecast --is_training 1 --root_path /data/Sajjan_Singh/spml/wavelet_seq_project/external/TSLib/dataset/weather --data_path weather.csv --model_id weather_FEDformer --model FEDformer --data

In [10]:
from pathlib import Path
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")

phase_a = ROOT / "results" / "tables" / "phaseA_all_datasets_metrics.csv"
phase_b = ROOT / "results" / "tables" / "phaseB_tslib_modern_metrics.csv"
master_out = ROOT / "results" / "tables" / "master_all_models_metrics.csv"

df_a = pd.read_csv(phase_a)
df_b = pd.read_csv(phase_b)

print("Phase B rows:", len(df_b))
print("\nPhase B status counts:")
print(df_b["status"].astype(str).value_counts(dropna=False).to_string())

usable_b = df_b[df_b["status"].astype(str).str.startswith(("ok_", "skipped_existing"))].copy()
print("\nUsable modern runs:", len(usable_b))

if len(usable_b) > 0:
    best_modern = (
        usable_b.sort_values(["dataset", "rmse", "mae", "mse"])
                .groupby("dataset", as_index=False)
                .first()
    )
    show_cols = [c for c in ["dataset", "model", "mse", "mae", "rmse", "use_amp", "batch_size", "d_model", "status"] if c in best_modern.columns]
    print("\nBest modern model per dataset:")
    print(best_modern[show_cols].to_string(index=False))

rename_map = {
    "mae": "test_mae",
    "mse": "test_mse",
    "rmse": "test_rmse",
}
usable_b = usable_b.rename(columns=rename_map)

common_cols = [
    "dataset", "model", "family",
    "seq_len", "pred_len",
    "params", "best_epoch", "train_seconds",
    "val_mse", "val_mae", "val_rmse",
    "test_mse", "test_mae", "test_rmse",
    "checkpoint", "history", "prediction_file",
]

for col in common_cols:
    if col not in df_a.columns:
        df_a[col] = None
    if col not in usable_b.columns:
        usable_b[col] = None

master = pd.concat([df_a[common_cols], usable_b[common_cols]], axis=0, ignore_index=True)
master = master.sort_values(["dataset", "family", "model"]).reset_index(drop=True)
master.to_csv(master_out, index=False)

print("\nSaved master file:", master_out)
print("Master rows:", len(master))

best_master = (
    master.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
          .groupby("dataset", as_index=False)
          .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
)

print("\nBest overall model per dataset:")
print(best_master.to_string(index=False))

best_master_out = ROOT / "results" / "tables" / "master_best_by_dataset.csv"
best_master.to_csv(best_master_out, index=False)
print("\nSaved:", best_master_out)

Phase B rows: 54

Phase B status counts:
status
skipped_existing                     42
excluded_incompatible_MS_protocol     9
ok_walltime_sec_1148.09               1
ok_walltime_sec_3195.47               1
ok_walltime_sec_7582.65               1

Usable modern runs: 45

Best modern model per dataset:
    dataset        model      mse      mae     rmse  use_amp  batch_size  d_model           status
electricity      WPMixer 0.218807 0.328796 0.467769     True          24      128 skipped_existing
      etth1      WPMixer 0.055708 0.178231 0.236025     True          48      192 skipped_existing
      etth2      WPMixer 0.132968 0.280923 0.364647     True          48      192 skipped_existing
      ettm1      WPMixer 0.027799 0.125130 0.166729     True          48      192 skipped_existing
      ettm2      WPMixer 0.063345 0.182120 0.251685     True          48      192 skipped_existing
   exchange      WPMixer 0.097021 0.227583 0.311482     True          48      192 skipped_existing
   

/tmp/ipykernel_1197340/2798227085.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master = pd.concat([df_a[common_cols], usable_b[common_cols]], axis=0, ignore_index=True)


In [12]:
%%bash
set -e
source ~/miniconda3/etc/profile.d/conda.sh 2>/dev/null || true
conda activate myenv 2>/dev/null || true

cd /data/Sajjan_Singh/spml/wavelet_seq_project
mkdir -p external results/logs/timemixer_official results/tables

if [ ! -d external/TimeMixer ]; then
  git clone https://github.com/kwuking/TimeMixer.git external/TimeMixer
fi

cd external/TimeMixer

# Fix pkg_resources missing (needed to build old packages)
pip install --upgrade setuptools --quiet

# Install compatible versions instead of the pinned ones in requirements.txt
# pandas==1.1.5 and matplotlib==3.4.3 fail to build on Python 3.10
pip install \
  einops==0.4.1 \
  "matplotlib>=3.5.0" \
  "numpy>=1.22.4,<2.0" \
  "pandas>=1.3.0" \
  reformer_pytorch \
  sympy \
  torch \
  --quiet

# Dataset folders expected by the official repo
mkdir -p dataset/ETT-small dataset/weather dataset/electricity dataset/traffic dataset/exchange_rate dataset/illness

# Symlink your datasets
ln -snf /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/etth1/ETTh1.csv       dataset/ETT-small/ETTh1.csv
ln -snf /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/etth2/ETTh2.csv       dataset/ETT-small/ETTh2.csv
ln -snf /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/ettm1/ETTm1.csv       dataset/ETT-small/ETTm1.csv
ln -snf /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/ettm2/ETTm2.csv       dataset/ETT-small/ETTm2.csv
ln -snf /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/weather/weather.csv   dataset/weather/weather.csv
ln -snf /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/electricity/electricity.csv dataset/electricity/electricity.csv
ln -snf /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/traffic/traffic.csv   dataset/traffic/traffic.csv
ln -snf /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/exchange/exchange_rate.csv  dataset/exchange_rate/exchange_rate.csv
ln -snf /data/Sajjan_Singh/spml/wavelet_seq_project/data/raw/ili/national_illness.csv    dataset/illness/national_illness.csv

echo "Official TimeMixer repo ready."

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
colt5-attention 0.11.1 requires einops>=0.8.0, but you have einops 0.4.1 which is incompatible.
colt5-attention 0.11.1 requires local-attention>=1.8.6, but you have local-attention 1.4.4 which is incompatible.
hyper-connections 0.4.9 requires einops>=0.8.1, but you have einops 0.4.1 which is incompatible.
torch-einops-utils 0.0.30 requires einops>=0.8.1, but you have einops 0.4.1 which is incompatible.


Official TimeMixer repo ready.


In [13]:
from pathlib import Path

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
script = ROOT / "scripts" / "run_timemixer_official.py"
script.parent.mkdir(parents=True, exist_ok=True)

script.write_text(r'''
from pathlib import Path
import csv
import time
import subprocess
import numpy as np

PROJECT_ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
TM_ROOT = PROJECT_ROOT / "external" / "TimeMixer"
OUT_CSV = PROJECT_ROOT / "results" / "tables" / "phaseB_timemixer_official_metrics.csv"
LOG_DIR = PROJECT_ROOT / "results" / "logs" / "timemixer_official"
LOG_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "etth1": {
        "data": "ETTh1", "root_path": str(TM_ROOT / "dataset" / "ETT-small"), "data_path": "ETTh1.csv",
        "freq": "h", "enc_in": 7, "seq_len": 96, "pred_len": 96, "label_len": 48
    },
    "etth2": {
        "data": "ETTh2", "root_path": str(TM_ROOT / "dataset" / "ETT-small"), "data_path": "ETTh2.csv",
        "freq": "h", "enc_in": 7, "seq_len": 96, "pred_len": 96, "label_len": 48
    },
    "ettm1": {
        "data": "ETTm1", "root_path": str(TM_ROOT / "dataset" / "ETT-small"), "data_path": "ETTm1.csv",
        "freq": "t", "enc_in": 7, "seq_len": 192, "pred_len": 96, "label_len": 96
    },
    "ettm2": {
        "data": "ETTm2", "root_path": str(TM_ROOT / "dataset" / "ETT-small"), "data_path": "ETTm2.csv",
        "freq": "t", "enc_in": 7, "seq_len": 192, "pred_len": 96, "label_len": 96
    },
    "weather": {
        "data": "custom", "root_path": str(TM_ROOT / "dataset" / "weather"), "data_path": "weather.csv",
        "freq": "t", "enc_in": 21, "seq_len": 288, "pred_len": 96, "label_len": 144
    },
    "electricity": {
        "data": "custom", "root_path": str(TM_ROOT / "dataset" / "electricity"), "data_path": "electricity.csv",
        "freq": "h", "enc_in": 321, "seq_len": 168, "pred_len": 96, "label_len": 84
    },
    "traffic": {
        "data": "custom", "root_path": str(TM_ROOT / "dataset" / "traffic"), "data_path": "traffic.csv",
        "freq": "h", "enc_in": 862, "seq_len": 168, "pred_len": 96, "label_len": 84
    },
    "exchange": {
        "data": "custom", "root_path": str(TM_ROOT / "dataset" / "exchange_rate"), "data_path": "exchange_rate.csv",
        "freq": "d", "enc_in": 8, "seq_len": 96, "pred_len": 96, "label_len": 48
    },
    "ili": {
        "data": "custom", "root_path": str(TM_ROOT / "dataset" / "illness"), "data_path": "national_illness.csv",
        "freq": "w", "enc_in": 7, "seq_len": 104, "pred_len": 24, "label_len": 52
    },
}

def append_row(row):
    file_exists = OUT_CSV.exists()
    with open(OUT_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

def find_result_dir(model_id):
    results_root = TM_ROOT / "results"
    candidates = sorted(
        [p for p in results_root.glob(f"*{model_id}_TimeMixer_*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    return candidates[0] if candidates else None

for dataset_name, cfg in DATASETS.items():
    model_id = f"{dataset_name}_TimeMixerOfficial"
    log_file = LOG_DIR / f"{dataset_name}__TimeMixerOfficial.log"

    existing_dir = find_result_dir(model_id)
    if existing_dir and (existing_dir / "metrics.npy").exists():
        metrics = np.load(existing_dir / "metrics.npy")
        mae, mse, rmse, mape, mspe = [float(x) for x in metrics.tolist()]
        append_row({
            "dataset": dataset_name,
            "model": "TimeMixerOfficial",
            "family": "modern_official",
            "seq_len": cfg["seq_len"],
            "label_len": cfg["label_len"],
            "pred_len": cfg["pred_len"],
            "enc_in": cfg["enc_in"],
            "freq": cfg["freq"],
            "target": "OT",
            "features": "MS",
            "mae": mae, "mse": mse, "rmse": rmse, "mape": mape, "mspe": mspe,
            "result_dir": str(existing_dir),
            "log_file": str(log_file),
            "status": "skipped_existing"
        })
        print(f"[SKIP] {dataset_name}")
        continue

    cmd = [
        "python", "-u", "run.py",
        "--task_name", "long_term_forecast",
        "--is_training", "1",
        "--root_path", cfg["root_path"],
        "--data_path", cfg["data_path"],
        "--model_id", model_id,
        "--model", "TimeMixer",
        "--data", cfg["data"],
        "--features", "MS",
        "--target", "OT",
        "--freq", cfg["freq"],
        "--seq_len", str(cfg["seq_len"]),
        "--label_len", str(cfg["label_len"]),
        "--pred_len", str(cfg["pred_len"]),
        "--enc_in", str(cfg["enc_in"]),
        "--dec_in", str(cfg["enc_in"]),
        "--c_out", str(cfg["enc_in"]),
        "--d_model", "192",
        "--d_ff", "768",
        "--e_layers", "2",
        "--d_layers", "1",
        "--dropout", "0.1",
        "--train_epochs", "50",
        "--patience", "10",
        "--batch_size", "32",
        "--learning_rate", "0.0001",
        "--num_workers", "8",
        "--itr", "1",
        "--des", "phaseB_timemixer_official",
        "--gpu", "0"
    ]

    print("\nRUN:", dataset_name)
    start = time.time()
    with open(log_file, "w") as lf:
        proc = subprocess.run(cmd, cwd=str(TM_ROOT), stdout=lf, stderr=subprocess.STDOUT, text=True)
    wall = round(time.time() - start, 2)

    result_dir = find_result_dir(model_id)
    metrics_path = result_dir / "metrics.npy" if result_dir else None

    if proc.returncode == 0 and metrics_path and metrics_path.exists():
        metrics = np.load(metrics_path)
        mae, mse, rmse, mape, mspe = [float(x) for x in metrics.tolist()]
        append_row({
            "dataset": dataset_name,
            "model": "TimeMixerOfficial",
            "family": "modern_official",
            "seq_len": cfg["seq_len"],
            "label_len": cfg["label_len"],
            "pred_len": cfg["pred_len"],
            "enc_in": cfg["enc_in"],
            "freq": cfg["freq"],
            "target": "OT",
            "features": "MS",
            "mae": mae, "mse": mse, "rmse": rmse, "mape": mape, "mspe": mspe,
            "result_dir": str(result_dir),
            "log_file": str(log_file),
            "status": f"ok_walltime_sec_{wall}"
        })
        print(f"[OK] {dataset_name} mse={mse:.6f} mae={mae:.6f}")
    else:
        append_row({
            "dataset": dataset_name,
            "model": "TimeMixerOfficial",
            "family": "modern_official",
            "seq_len": cfg["seq_len"],
            "label_len": cfg["label_len"],
            "pred_len": cfg["pred_len"],
            "enc_in": cfg["enc_in"],
            "freq": cfg["freq"],
            "target": "OT",
            "features": "MS",
            "mae": None, "mse": None, "rmse": None, "mape": None, "mspe": None,
            "result_dir": str(result_dir) if result_dir else None,
            "log_file": str(log_file),
            "status": f"failed_returncode_{proc.returncode}"
        })
        print(f"[FAIL] {dataset_name} -> {log_file}")
''')

print("Wrote:", script)

Wrote: /data/Sajjan_Singh/spml/wavelet_seq_project/scripts/run_timemixer_official.py


In [14]:
%%bash
set -e
source ~/miniconda3/etc/profile.d/conda.sh 2>/dev/null || true
conda activate myenv 2>/dev/null || true

cd /data/Sajjan_Singh/spml/wavelet_seq_project
python scripts/run_timemixer_official.py


RUN: etth1
[FAIL] etth1 -> /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/timemixer_official/etth1__TimeMixerOfficial.log

RUN: etth2
[FAIL] etth2 -> /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/timemixer_official/etth2__TimeMixerOfficial.log

RUN: ettm1
[FAIL] ettm1 -> /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/timemixer_official/ettm1__TimeMixerOfficial.log

RUN: ettm2
[FAIL] ettm2 -> /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/timemixer_official/ettm2__TimeMixerOfficial.log

RUN: weather
[FAIL] weather -> /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/timemixer_official/weather__TimeMixerOfficial.log

RUN: electricity
[FAIL] electricity -> /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/timemixer_official/electricity__TimeMixerOfficial.log

RUN: traffic
[FAIL] traffic -> /data/Sajjan_Singh/spml/wavelet_seq_project/results/logs/timemixer_official/traffic__TimeMixerOfficial.log

RUN: exchange
[FAIL] exchange -> /data

In [15]:
from pathlib import Path
import pandas as pd

csv_path = Path("/data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseB_timemixer_official_metrics.csv")
df = pd.read_csv(csv_path)

print("Rows:", len(df))
print("\nStatus counts:")
print(df["status"].astype(str).value_counts(dropna=False).to_string())

print("\nUsable runs:")
ok = df[df["status"].astype(str).str.startswith(("ok_", "skipped_existing"))]
print(ok[["dataset", "model", "mse", "mae", "rmse", "status"]].to_string(index=False))

Rows: 9

Status counts:
status
failed_returncode_1    9

Usable runs:
Empty DataFrame
Columns: [dataset, model, mse, mae, rmse, status]
Index: []


In [16]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
PROCESSED = ROOT / "data" / "processed"
SPLITS = ROOT / "data" / "splits"
OUT_CSV = ROOT / "results" / "tables" / "phaseA_classical_univariate_metrics.csv"

DATASETS = ["etth1", "etth2", "ettm1", "ettm2", "weather", "electricity", "traffic", "exchange", "ili"]

SEASONAL_PERIODS = {
    "etth1": 24,
    "etth2": 24,
    "ettm1": 96,
    "ettm2": 96,
    "weather": 144,
    "electricity": 24,
    "traffic": 24,
    "exchange": 7,
    "ili": 52,
}

def mse(y, yhat):
    return float(np.mean((y - yhat) ** 2))

def mae(y, yhat):
    return float(np.mean(np.abs(y - yhat)))

rows = []

for ds in DATASETS:
    npz_path = PROCESSED / ds / f"{ds}_prepared.npz"
    split_path = SPLITS / f"{ds}_splits.json"

    arr = np.load(npz_path, allow_pickle=True)
    with open(split_path, "r") as f:
        splits = json.load(f)

    values_raw = arr["values_raw"]
    target_idx = int(arr["target_idx"][0])
    y = values_raw[:, target_idx].astype(float)

    tr0, tr1 = splits["train_start"], splits["train_end"]
    va0, va1 = splits["val_start"], splits["val_end"]
    te0, te1 = splits["test_start"], splits["test_end"]

    y_train = y[tr0:tr1]
    y_val = y[va0:va1]
    y_test = y[te0:te1]

    best_name = None
    best_val_mse = np.inf
    best_forecast_test = None

    # ETS
    try:
        sp = SEASONAL_PERIODS[ds]
        seasonal = "add" if len(y_train) > 2 * sp else None
        trend = "add"
        ets_model = ExponentialSmoothing(
            y_train,
            trend=trend,
            seasonal=seasonal,
            seasonal_periods=sp if seasonal else None,
            initialization_method="estimated"
        ).fit(optimized=True)

        val_pred = ets_model.forecast(len(y_val))
        val_mse = mse(y_val, val_pred)

        trainval = y[:va1]
        ets_refit = ExponentialSmoothing(
            trainval,
            trend=trend,
            seasonal=seasonal,
            seasonal_periods=sp if seasonal else None,
            initialization_method="estimated"
        ).fit(optimized=True)
        test_pred = ets_refit.forecast(len(y_test))

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_name = "ETS"
            best_forecast_test = np.asarray(test_pred, dtype=float)
    except Exception as e:
        print(f"{ds} | ETS failed: {e}")

    # Small ARIMA grid
    arima_grid = [(1,0,0), (2,0,0), (1,1,0), (1,1,1), (2,1,2)]
    for order in arima_grid:
        try:
            model = ARIMA(y_train, order=order).fit()
            val_pred = model.forecast(len(y_val))
            val_mse = mse(y_val, val_pred)

            trainval = y[:va1]
            refit = ARIMA(trainval, order=order).fit()
            test_pred = refit.forecast(len(y_test))

            if val_mse < best_val_mse:
                best_val_mse = val_mse
                best_name = f"ARIMA{order}"
                best_forecast_test = np.asarray(test_pred, dtype=float)
        except Exception:
            pass

    if best_forecast_test is None:
        print(f"{ds} | classical baseline failed completely")
        continue

    row = {
        "dataset": ds,
        "model": best_name,
        "family": "classical_univariate",
        "test_mse": mse(y_test, best_forecast_test),
        "test_mae": mae(y_test, best_forecast_test),
        "test_rmse": float(np.sqrt(mse(y_test, best_forecast_test))),
    }
    rows.append(row)
    print(ds, row)

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)
df

/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


etth1 {'dataset': 'etth1', 'model': 'ARIMA(2, 0, 0)', 'family': 'classical_univariate', 'test_mse': 147.85549204307023, 'test_mae': 11.557632119329778, 'test_rmse': 12.159584369667831}


/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


etth2 {'dataset': 'etth2', 'model': 'ETS', 'family': 'classical_univariate', 'test_mse': 44.802213297079675, 'test_mae': 5.540183988381152, 'test_rmse': 6.693445547479988}


/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


...

/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/tsa/holtwinters/model.py:903: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


ettm2 {'dataset': 'ettm2', 'model': 'ARIMA(1, 1, 0)', 'family': 'classical_univariate', 'test_mse': 76.15465508527372, 'test_mae': 7.2671380027846935, 'test_rmse': 8.726663456629556}
weather {'dataset': 'weather', 'model': 'ARIMA(1, 0, 0)', 'family': 'classical_univariate', 'test_mse': 852.186969637021, 'test_mae': 24.24677252073151, 'test_rmse': 29.1922416000728}


/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


electricity {'dataset': 'electricity', 'model': 'ARIMA(2, 1, 2)', 'family': 'classical_univariate', 'test_mse': 352654.2653468062, 'test_mae': 450.14840805930424, 'test_rmse': 593.8470049994411}
traffic {'dataset': 'traffic', 'model': 'ETS', 'family': 'classical_univariate', 'test_mse': 0.00019946135463473242, 'test_mae': 0.011728574412291732, 'test_rmse': 0.014123078794467317}
exchange {'dataset': 'exchange', 'model': 'ETS', 'family': 'classical_univariate', 'test_mse': 0.01600926780307293, 'test_mae': 0.0964973345812004, 'test_rmse': 0.12652773531156294}
ili {'dataset': 'ili', 'model': 'ETS', 'family': 'classical_univariate', 'test_mse': 149011774356.46722, 'test_mae': 336052.78457528097, 'test_rmse': 386020.43256344245}

Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_classical_univariate_metrics.csv


,dataset,model,family,test_mse,test_mae,test_rmse
0,etth1,"ARIMA(2, 0, 0)",classical_univariate,1.478555e+02,11.557632,12.159584
1,etth2,ETS,classical_univariate,4.480221e+01,5.540184,6.693446
2,ettm1,"ARIMA(1, 0, 0)",classical_univariate,1.477189e+02,11.549256,12.153965
3,ettm2,"ARIMA(1, 1, 0)",classical_univariate,7.615466e+01,7.267138,8.726663
4,weather,"ARIMA(1, 0, 0)",classical_univariate,8.521870e+02,24.246773,29.192242
5,electricity,"ARIMA(2, 1, 2)",classical_univariate,3.526543e+05,450.148408,593.847005
6,traffic,ETS,classical_univariate,1.994614e-04,0.011729,0.014123
7,exchange,ETS,classical_univariate,1.600927e-02,0.096497,0.126528
8,ili,ETS,classical_univariate,1.490118e+11,336052.784575,386020.432563


In [17]:
from pathlib import Path
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
master_path = ROOT / "results" / "tables" / "master_all_models_metrics.csv"
classical_path = ROOT / "results" / "tables" / "phaseA_classical_univariate_metrics.csv"

master = pd.read_csv(master_path)
classical = pd.read_csv(classical_path)

for col in master.columns:
    if col not in classical.columns:
        classical[col] = None

classical = classical[master.columns]
merged = pd.concat([master, classical], ignore_index=True)
merged = merged.sort_values(["dataset", "family", "model"]).reset_index(drop=True)

merged.to_csv(master_path, index=False)
print("Updated master:", master_path)
print("Rows:", len(merged))

Updated master: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/master_all_models_metrics.csv
Rows: 108


In [18]:
import gc
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
PROCESSED_ROOT = ROOT / "data" / "processed"
SPLIT_ROOT = ROOT / "data" / "splits"
OUT_CSV = ROOT / "results" / "tables" / "phaseA_mLSTM_metrics.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_RUN_CONFIG = {
    "etth1": {"seq_len": 96, "pred_len": 96, "batch_size": 256, "epochs": 25},
    "etth2": {"seq_len": 96, "pred_len": 96, "batch_size": 256, "epochs": 25},
    "ettm1": {"seq_len": 192, "pred_len": 96, "batch_size": 256, "epochs": 25},
    "ettm2": {"seq_len": 192, "pred_len": 96, "batch_size": 256, "epochs": 25},
    "weather": {"seq_len": 288, "pred_len": 96, "batch_size": 192, "epochs": 25},
    "electricity": {"seq_len": 168, "pred_len": 96, "batch_size": 128, "epochs": 25},
    "traffic": {"seq_len": 168, "pred_len": 96, "batch_size": 96, "epochs": 25},
    "exchange": {"seq_len": 96, "pred_len": 96, "batch_size": 256, "epochs": 25},
    "ili": {"seq_len": 104, "pred_len": 24, "batch_size": 64, "epochs": 30},
}

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mse = float(np.mean((y_true - y_pred) ** 2))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(mse))
    return {"mse": mse, "mae": mae, "rmse": rmse}

def load_bundle(dataset_name):
    npz_path = PROCESSED_ROOT / dataset_name / f"{dataset_name}_prepared.npz"
    split_path = SPLIT_ROOT / f"{dataset_name}_splits.json"
    data = np.load(npz_path, allow_pickle=True)
    with open(split_path, "r") as f:
        splits = json.load(f)
    return {
        "values_raw": data["values_raw"].astype(np.float32),
        "values_scaled": data["values_scaled"].astype(np.float32),
        "target_idx": int(data["target_idx"][0]),
        "scaler_mean": data["scaler_mean"].astype(np.float32),
        "scaler_std": data["scaler_std"].astype(np.float32),
        "splits": {k: int(v) for k, v in splits.items()},
    }

class ForecastWindowDataset(Dataset):
    def __init__(self, bundle, split_name, seq_len, pred_len):
        self.values_raw = bundle["values_raw"]
        self.values_scaled = bundle["values_scaled"]
        self.target_idx = bundle["target_idx"]
        s = bundle["splits"]

        if split_name == "train":
            b1, b2 = s["train_start"], s["train_end"]
        elif split_name == "val":
            b1, b2 = max(0, s["val_start"] - seq_len), s["val_end"]
        else:
            b1, b2 = max(0, s["test_start"] - seq_len), s["test_end"]

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.border1 = b1
        self.border2 = b2
        self.length = b2 - b1 - seq_len - pred_len + 1

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        s = self.border1 + idx
        x = self.values_scaled[s:s+self.seq_len]
        y_scaled = self.values_scaled[s+self.seq_len:s+self.seq_len+self.pred_len, self.target_idx]
        y_raw = self.values_raw[s+self.seq_len:s+self.seq_len+self.pred_len, self.target_idx]
        return {
            "x_scaled": torch.tensor(x, dtype=torch.float32),
            "y_scaled": torch.tensor(y_scaled, dtype=torch.float32),
            "y_raw": torch.tensor(y_raw, dtype=torch.float32),
        }

def make_loaders(bundle, seq_len, pred_len, batch_size):
    train_ds = ForecastWindowDataset(bundle, "train", seq_len, pred_len)
    val_ds = ForecastWindowDataset(bundle, "val", seq_len, pred_len)
    test_ds = ForecastWindowDataset(bundle, "test", seq_len, pred_len)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available()),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available()),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available()),
    )

class LearnableWaveletDecomp(nn.Module):
    def __init__(self, channels):
        super().__init__()
        sqrt2 = math.sqrt(2.0)
        lp = torch.tensor([1/sqrt2, 1/sqrt2], dtype=torch.float32)
        hp = torch.tensor([1/sqrt2, -1/sqrt2], dtype=torch.float32)

        self.low = nn.Conv1d(channels, channels, kernel_size=2, stride=2, groups=channels, bias=False)
        self.high = nn.Conv1d(channels, channels, kernel_size=2, stride=2, groups=channels, bias=False)

        with torch.no_grad():
            self.low.weight[:] = lp.view(1, 1, 2).repeat(channels, 1, 1)
            self.high.weight[:] = hp.view(1, 1, 2).repeat(channels, 1, 1)

    def forward(self, x):
        # x: [B, L, D]
        x = x.transpose(1, 2)  # [B, D, L]
        if x.shape[-1] % 2 == 1:
            x = torch.cat([x, x[..., -1:]], dim=-1)

        low = self.low(x)
        high = self.high(x)

        low_up = low.repeat_interleave(2, dim=-1)[..., :x.shape[-1]]
        high_up = high.repeat_interleave(2, dim=-1)[..., :x.shape[-1]]

        return low_up.transpose(1, 2), high_up.transpose(1, 2)

class mLSTMBaseline(nn.Module):
    def __init__(self, input_dim, pred_len, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.decomp = LearnableWaveletDecomp(input_dim)

        total_dim = input_dim * 3
        self.proj = nn.Linear(total_dim, 128)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, pred_len),
        )

    def forward(self, x):
        low, high = self.decomp(x)
        z = torch.cat([x, low, high], dim=-1)
        z = self.proj(z)
        out, _ = self.lstm(z)
        h = out[:, -1, :]
        return self.head(h)

def train_eval_one(dataset_name, cfg):
    set_seed(42)
    bundle = load_bundle(dataset_name)
    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    train_loader, val_loader, test_loader = make_loaders(
        bundle, cfg["seq_len"], cfg["pred_len"], cfg["batch_size"]
    )

    model = mLSTMBaseline(
        input_dim=bundle["values_scaled"].shape[1],
        pred_len=cfg["pred_len"]
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.MSELoss()

    best_val = float("inf")
    patience = 5
    best_state = None

    def eval_loader(loader):
        model.eval()
        preds, trues = [], []
        loss_sum, n = 0.0, 0
        with torch.no_grad():
            for batch in loader:
                x = batch["x_scaled"].to(DEVICE)
                y_scaled = batch["y_scaled"].to(DEVICE)
                y_raw = batch["y_raw"].cpu().numpy()

                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    pred_scaled = model(x)
                    loss = criterion(pred_scaled, y_scaled)

                pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
                preds.append(pred_raw)
                trues.append(y_raw)
                loss_sum += loss.item() * x.size(0)
                n += x.size(0)

        preds = np.concatenate(preds, axis=0)
        trues = np.concatenate(trues, axis=0)
        m = compute_metrics(trues.reshape(-1), preds.reshape(-1))
        m["scaled_loss"] = loss_sum / max(n, 1)
        return m

    for epoch in range(cfg["epochs"]):
        model.train()
        for batch in train_loader:
            x = batch["x_scaled"].to(DEVICE)
            y = batch["y_scaled"].to(DEVICE)

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                pred = model(x)
                loss = criterion(pred, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        val_m = eval_loader(val_loader)
        if val_m["mse"] < best_val:
            best_val = val_m["mse"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 5
        else:
            patience -= 1
            if patience == 0:
                break

    model.load_state_dict(best_state)
    test_m = eval_loader(test_loader)

    row = {
        "dataset": dataset_name,
        "model": "mLSTM_reproduced",
        "family": "wavelet_learnable",
        "test_mse": test_m["mse"],
        "test_mae": test_m["mae"],
        "test_rmse": test_m["rmse"],
    }
    del model
    torch.cuda.empty_cache()
    gc.collect()
    return row

rows = []
for ds, cfg in DATASET_RUN_CONFIG.items():
    print("Running", ds)
    rows.append(train_eval_one(ds, cfg))

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)
df

Running etth1
Running etth2
Running ettm1
Running ettm2
Running weather
Running electricity
Running traffic
Running exchange
Running ili

Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_mLSTM_metrics.csv


,dataset,model,family,test_mse,test_mae,test_rmse
0,etth1,mLSTM_reproduced,wavelet_learnable,2.821726e+01,4.742370,5.311992
1,etth2,mLSTM_reproduced,wavelet_learnable,1.083756e+02,8.162314,10.410360
2,ettm1,mLSTM_reproduced,wavelet_learnable,1.176870e+01,2.885064,3.430554
3,ettm2,mLSTM_reproduced,wavelet_learnable,8.043043e+01,7.465128,8.968301
4,weather,mLSTM_reproduced,wavelet_learnable,4.822665e+02,16.498678,21.960567
5,electricity,mLSTM_reproduced,wavelet_learnable,1.433377e+05,280.085881,378.599674
6,traffic,mLSTM_reproduced,wavelet_learnable,8.359086e-05,0.005795,0.009143
7,exchange,mLSTM_reproduced,wavelet_learnable,1.056295e-02,0.090872,0.102776
8,ili,mLSTM_reproduced,wavelet_learnable,1.972861e+11,381572.141406,444169.001051


In [19]:
from pathlib import Path
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
master_path = ROOT / "results" / "tables" / "master_all_models_metrics.csv"
mlstm_path = ROOT / "results" / "tables" / "phaseA_mLSTM_metrics.csv"

master = pd.read_csv(master_path)
mlstm = pd.read_csv(mlstm_path)

for col in master.columns:
    if col not in mlstm.columns:
        mlstm[col] = None

mlstm = mlstm[master.columns]
master = pd.concat([master, mlstm], ignore_index=True)
master = master.sort_values(["dataset", "family", "model"]).reset_index(drop=True)
master.to_csv(master_path, index=False)

print("Updated master:", master_path)
print("Rows:", len(master))

Updated master: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/master_all_models_metrics.csv
Rows: 117


In [20]:
from pathlib import Path
import pandas as pd

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")

checks = {
    "phase_a_main": ROOT / "results" / "tables" / "phaseA_all_datasets_metrics.csv",
    "phase_b_modern": ROOT / "results" / "tables" / "phaseB_tslib_modern_metrics.csv",
    "timemixer_official": ROOT / "results" / "tables" / "phaseB_timemixer_official_metrics.csv",
    "classical_univariate": ROOT / "results" / "tables" / "phaseA_classical_univariate_metrics.csv",
    "mlstm_baseline": ROOT / "results" / "tables" / "phaseA_mLSTM_metrics.csv",
    "master": ROOT / "results" / "tables" / "master_all_models_metrics.csv",
    "master_best": ROOT / "results" / "tables" / "master_best_by_dataset.csv",
}

print("=" * 100)
print("FILE EXISTENCE CHECK")
print("=" * 100)
for name, path in checks.items():
    print(f"{name:22s} -> {'FOUND' if path.exists() else 'MISSING'} | {path}")

print("\n" + "=" * 100)
print("CONTENT CHECK")
print("=" * 100)

def safe_read_csv(path):
    if path.exists():
        return pd.read_csv(path)
    return None

df_phase_a = safe_read_csv(checks["phase_a_main"])
df_phase_b = safe_read_csv(checks["phase_b_modern"])
df_tm = safe_read_csv(checks["timemixer_official"])
df_classical = safe_read_csv(checks["classical_univariate"])
df_mlstm = safe_read_csv(checks["mlstm_baseline"])
df_master = safe_read_csv(checks["master"])

if df_phase_a is not None:
    print(f"\nPhase A rows: {len(df_phase_a)}")
    print("Phase A datasets:", sorted(df_phase_a["dataset"].dropna().unique().tolist()))
    print("Phase A models:", sorted(df_phase_a["model"].dropna().unique().tolist()))

if df_phase_b is not None:
    print(f"\nPhase B rows: {len(df_phase_b)}")
    if "status" in df_phase_b.columns:
        print("Phase B status counts:")
        print(df_phase_b["status"].astype(str).value_counts(dropna=False).to_string())
    print("Phase B models:", sorted(df_phase_b["model"].dropna().unique().tolist()))

if df_tm is not None:
    print(f"\nTimeMixer official rows: {len(df_tm)}")
    if "status" in df_tm.columns:
        print("TimeMixer official status counts:")
        print(df_tm["status"].astype(str).value_counts(dropna=False).to_string())

if df_classical is not None:
    print(f"\nClassical univariate rows: {len(df_classical)}")
    print("Classical models:", sorted(df_classical["model"].dropna().unique().tolist()))

if df_mlstm is not None:
    print(f"\nmLSTM rows: {len(df_mlstm)}")
    print("mLSTM models:", sorted(df_mlstm["model"].dropna().unique().tolist()))

if df_master is not None:
    print(f"\nMaster rows: {len(df_master)}")
    print("Master datasets:", sorted(df_master["dataset"].dropna().unique().tolist()))
    print("Master models:", sorted(df_master["model"].dropna().unique().tolist()))

    best = (
        df_master.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
                 .groupby("dataset", as_index=False)
                 .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
    )
    print("\nBest overall model per dataset:")
    print(best.to_string(index=False))

print("\n" + "=" * 100)
print("DECISION CHECK")
print("=" * 100)

baseline_ok = (
    df_phase_a is not None and len(df_phase_a) >= 54 and
    df_classical is not None and len(df_classical) >= 9 and
    df_mlstm is not None and len(df_mlstm) >= 9
)

modern_ok = (
    df_phase_b is not None and len(df_phase_b) >= 54
)

master_ok = (
    df_master is not None and
    "dataset" in df_master.columns and
    "model" in df_master.columns and
    "test_rmse" in df_master.columns
)

if baseline_ok and modern_ok and master_ok:
    print("STATUS: Benchmark ladder is sufficiently complete to move to 03_analysis_and_plots.ipynb")
else:
    print("STATUS: Do NOT move yet. Some missing-baseline or master-table pieces are still incomplete.")

FILE EXISTENCE CHECK
phase_a_main           -> FOUND | /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_all_datasets_metrics.csv
phase_b_modern         -> FOUND | /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseB_tslib_modern_metrics.csv
timemixer_official     -> FOUND | /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseB_timemixer_official_metrics.csv
classical_univariate   -> FOUND | /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_classical_univariate_metrics.csv
mlstm_baseline         -> FOUND | /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_mLSTM_metrics.csv
master                 -> FOUND | /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/master_all_models_metrics.csv
master_best            -> FOUND | /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/master_best_by_dataset.csv

CONTENT CHECK

Phase A rows: 54
Phase A datasets: ['electricity', 'etth1', 'etth2', 'ettm1', 'ettm2', '

In [1]:
# ============================================
# NEW FINAL CELL: Save raw-unit predictions for ARIMA/ETS and mLSTM_reproduced
# ============================================

import gc
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
PROCESSED_ROOT = ROOT / "data" / "processed"
SPLIT_ROOT = ROOT / "data" / "splits"
PRED_ROOT = ROOT / "results" / "predictions"
TABLES_ROOT = ROOT / "results" / "tables"
PRED_ROOT.mkdir(parents=True, exist_ok=True)
TABLES_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

PHASEA_CFG = {
    "etth1": {"seq_len": 96, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 24},
    "etth2": {"seq_len": 96, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 24},
    "ettm1": {"seq_len": 192, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 96},
    "ettm2": {"seq_len": 192, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 96},
    "weather": {"seq_len": 288, "pred_len": 96, "batch_size": 192, "epochs": 25, "seasonality": 144},
    "electricity": {"seq_len": 168, "pred_len": 96, "batch_size": 128, "epochs": 25, "seasonality": 24},
    "traffic": {"seq_len": 168, "pred_len": 96, "batch_size": 96, "epochs": 25, "seasonality": 24},
    "exchange": {"seq_len": 96, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 7},
    "ili": {"seq_len": 104, "pred_len": 24, "batch_size": 64, "epochs": 30, "seasonality": 52},
}

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mse(y, yhat):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    return float(np.mean((y - yhat) ** 2))

def mae(y, yhat):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    return float(np.sqrt(mse(y, yhat)))

def load_bundle(dataset_name):
    npz_path = PROCESSED_ROOT / dataset_name / f"{dataset_name}_prepared.npz"
    split_path = SPLIT_ROOT / f"{dataset_name}_splits.json"
    arr = np.load(npz_path, allow_pickle=True)
    with open(split_path, "r") as f:
        splits = json.load(f)
    return {
        "values_raw": arr["values_raw"].astype(np.float32),
        "values_scaled": arr["values_scaled"].astype(np.float32),
        "target_idx": int(arr["target_idx"][0]),
        "scaler_mean": arr["scaler_mean"].astype(np.float32),
        "scaler_std": arr["scaler_std"].astype(np.float32),
        "splits": {k: int(v) for k, v in splits.items()},
    }

class ForecastWindowDataset(Dataset):
    def __init__(self, bundle, split_name, seq_len, pred_len):
        self.values_raw = bundle["values_raw"]
        self.values_scaled = bundle["values_scaled"]
        self.target_idx = bundle["target_idx"]
        s = bundle["splits"]

        if split_name == "train":
            b1, b2 = s["train_start"], s["train_end"]
        elif split_name == "val":
            b1, b2 = max(0, s["val_start"] - seq_len), s["val_end"]
        elif split_name == "test":
            b1, b2 = max(0, s["test_start"] - seq_len), s["test_end"]
        else:
            raise ValueError(split_name)

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.border1 = b1
        self.border2 = b2
        self.length = b2 - b1 - seq_len - pred_len + 1

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        s = self.border1 + idx
        x_scaled = self.values_scaled[s:s+self.seq_len]
        y_scaled = self.values_scaled[s+self.seq_len:s+self.seq_len+self.pred_len, self.target_idx]
        y_raw = self.values_raw[s+self.seq_len:s+self.seq_len+self.pred_len, self.target_idx]
        return {
            "x_scaled": torch.tensor(x_scaled, dtype=torch.float32),
            "y_scaled": torch.tensor(y_scaled, dtype=torch.float32),
            "y_raw": torch.tensor(y_raw, dtype=torch.float32),
        }

def make_loaders(bundle, seq_len, pred_len, batch_size):
    train_ds = ForecastWindowDataset(bundle, "train", seq_len, pred_len)
    val_ds = ForecastWindowDataset(bundle, "val", seq_len, pred_len)
    test_ds = ForecastWindowDataset(bundle, "test", seq_len, pred_len)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available()),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available()),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available()),
    )

class LearnableWaveletDecomp(nn.Module):
    def __init__(self, channels):
        super().__init__()
        sqrt2 = math.sqrt(2.0)
        lp = torch.tensor([1/sqrt2, 1/sqrt2], dtype=torch.float32)
        hp = torch.tensor([1/sqrt2, -1/sqrt2], dtype=torch.float32)

        self.low = nn.Conv1d(channels, channels, kernel_size=2, stride=2, groups=channels, bias=False)
        self.high = nn.Conv1d(channels, channels, kernel_size=2, stride=2, groups=channels, bias=False)

        with torch.no_grad():
            self.low.weight[:] = lp.view(1, 1, 2).repeat(channels, 1, 1)
            self.high.weight[:] = hp.view(1, 1, 2).repeat(channels, 1, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        if x.shape[-1] % 2 == 1:
            x = torch.cat([x, x[..., -1:]], dim=-1)

        low = self.low(x)
        high = self.high(x)

        low_up = low.repeat_interleave(2, dim=-1)[..., :x.shape[-1]]
        high_up = high.repeat_interleave(2, dim=-1)[..., :x.shape[-1]]

        return low_up.transpose(1, 2), high_up.transpose(1, 2)

class MLSTMBaseline(nn.Module):
    def __init__(self, input_dim, pred_len, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.decomp = LearnableWaveletDecomp(input_dim)
        self.proj = nn.Linear(input_dim * 3, 128)
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, pred_len),
        )

    def forward(self, x):
        low, high = self.decomp(x)
        z = torch.cat([x, low, high], dim=-1)
        z = self.proj(z)
        out, _ = self.lstm(z)
        return self.head(out[:, -1, :])

@torch.no_grad()
def predict_raw(model, loader, target_mean, target_std):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        x = batch["x_scaled"].to(DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred_scaled = model(x)
        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
    return np.concatenate(preds, axis=0), np.concatenate(trues, axis=0)

def run_classical_and_save(dataset_name):
    bundle = load_bundle(dataset_name)
    cfg = PHASEA_CFG[dataset_name]
    s = bundle["splits"]
    target_idx = bundle["target_idx"]

    y = bundle["values_raw"][:, target_idx].astype(float)

    tr0, tr1 = s["train_start"], s["train_end"]
    va0, va1 = s["val_start"], s["val_end"]
    te0, te1 = s["test_start"], s["test_end"]

    y_train = y[tr0:tr1]
    y_val = y[va0:va1]
    y_test = y[te0:te1]

    best_name = None
    best_val_mse = np.inf
    best_test_pred = None

    # ETS
    try:
        sp = cfg["seasonality"]
        seasonal = "add" if len(y_train) > 2 * sp else None
        ets = ExponentialSmoothing(
            y_train,
            trend="add",
            seasonal=seasonal,
            seasonal_periods=sp if seasonal else None,
            initialization_method="estimated"
        ).fit(optimized=True)
        val_pred = np.asarray(ets.forecast(len(y_val)), dtype=float)
        val_m = mse(y_val, val_pred)

        trainval = y[:va1]
        ets2 = ExponentialSmoothing(
            trainval,
            trend="add",
            seasonal=seasonal,
            seasonal_periods=sp if seasonal else None,
            initialization_method="estimated"
        ).fit(optimized=True)
        test_pred = np.asarray(ets2.forecast(len(y_test)), dtype=float)

        if val_m < best_val_mse:
            best_val_mse = val_m
            best_name = "ETS"
            best_test_pred = test_pred
    except Exception as e:
        print(dataset_name, "ETS failed:", e)

    # ARIMA grid
    for order in [(1,0,0), (2,0,0), (1,1,0), (1,1,1), (2,1,2)]:
        try:
            ar = ARIMA(y_train, order=order).fit()
            val_pred = np.asarray(ar.forecast(len(y_val)), dtype=float)
            val_m = mse(y_val, val_pred)

            trainval = y[:va1]
            ar2 = ARIMA(trainval, order=order).fit()
            test_pred = np.asarray(ar2.forecast(len(y_test)), dtype=float)

            if val_m < best_val_mse:
                best_val_mse = val_m
                best_name = f"ARIMA{order}"
                best_test_pred = test_pred
        except Exception:
            pass

    if best_test_pred is None:
        raise RuntimeError(f"{dataset_name}: ARIMA/ETS all failed")

    pred_dir = PRED_ROOT / dataset_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    pred_path = pred_dir / "classical_univariate_test_predictions.npz"
    np.savez_compressed(
        pred_path,
        preds=best_test_pred.reshape(-1, 1),
        trues=y_test.reshape(-1, 1),
        seq_len=cfg["seq_len"],
        pred_len=cfg["pred_len"],
        model=best_name,
    )

    return {
        "dataset": dataset_name,
        "model": best_name,
        "family": "classical_univariate",
        "seq_len": cfg["seq_len"],
        "pred_len": cfg["pred_len"],
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": mse(y_test, best_test_pred),
        "test_mae": mae(y_test, best_test_pred),
        "test_rmse": rmse(y_test, best_test_pred),
    }

def run_mlstm_and_save(dataset_name):
    set_seed(42)
    bundle = load_bundle(dataset_name)
    cfg = PHASEA_CFG[dataset_name]

    seq_len = cfg["seq_len"]
    pred_len = cfg["pred_len"]
    batch_size = cfg["batch_size"]
    epochs = cfg["epochs"]

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    train_loader, val_loader, test_loader = make_loaders(bundle, seq_len, pred_len, batch_size)
    model = MLSTMBaseline(
        input_dim=bundle["values_scaled"].shape[1],
        pred_len=pred_len
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.MSELoss()

    best_val = np.inf
    patience = 5
    best_state = None

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            x = batch["x_scaled"].to(DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                pred = model(x)
                loss = criterion(pred, y)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        # quick val
        val_preds, val_trues = predict_raw(model, val_loader, target_mean, target_std)
        val_rmse = rmse(val_trues.reshape(-1), val_preds.reshape(-1))

        if val_rmse < best_val:
            best_val = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 5
        else:
            patience -= 1
            if patience == 0:
                break

    model.load_state_dict(best_state)

    test_preds, test_trues = predict_raw(model, test_loader, target_mean, target_std)

    pred_dir = PRED_ROOT / dataset_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    pred_path = pred_dir / "mLSTM_reproduced_test_predictions.npz"
    np.savez_compressed(
        pred_path,
        preds=test_preds,
        trues=test_trues,
        seq_len=seq_len,
        pred_len=pred_len,
    )

    row = {
        "dataset": dataset_name,
        "model": "mLSTM_reproduced",
        "family": "wavelet_learnable",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": mse(test_trues.reshape(-1), test_preds.reshape(-1)),
        "test_mae": mae(test_trues.reshape(-1), test_preds.reshape(-1)),
        "test_rmse": rmse(test_trues.reshape(-1), test_preds.reshape(-1)),
    }

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row

registry_rows = []

for ds in PHASEA_CFG:
    print("\n" + "=" * 100)
    print("DATASET:", ds)
    print("=" * 100)

    classical_row = run_classical_and_save(ds)
    print("Saved classical predictions:", classical_row["prediction_file"])
    registry_rows.append(classical_row)

    mlstm_row = run_mlstm_and_save(ds)
    print("Saved mLSTM predictions:", mlstm_row["prediction_file"])
    registry_rows.append(mlstm_row)

registry_df = pd.DataFrame(registry_rows)
registry_path = TABLES_ROOT / "phaseA_extra_prediction_registry.csv"
registry_df.to_csv(registry_path, index=False)

print("\nSaved:", registry_path)
display(registry_df)


DATASET: etth1


/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Saved classical predictions: results/predictions/etth1/classical_univariate_test_predictions.npz


/tmp/ipykernel_2175268/3784396039.py:309: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_2175268/3784396039.py:323: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_2175268/3784396039.py:185: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Saved mLSTM predictions: results/predictions/etth1/mLSTM_reproduced_test_predictions.npz

DATASET: etth2


...

/tmp/ipykernel_2175268/3784396039.py:309: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_2175268/3784396039.py:323: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_2175268/3784396039.py:185: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Saved mLSTM predictions: results/predictions/exchange/mLSTM_reproduced_test_predictions.npz

DATASET: ili
Saved classical predictions: results/predictions/ili/classical_univariate_test_predictions.npz


/tmp/ipykernel_2175268/3784396039.py:309: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_2175268/3784396039.py:323: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_2175268/3784396039.py:185: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Saved mLSTM predictions: results/predictions/ili/mLSTM_reproduced_test_predictions.npz

Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_extra_prediction_registry.csv


,dataset,model,family,seq_len,pred_len,prediction_file,test_mse,test_mae,test_rmse
0,etth1,"ARIMA(2, 0, 0)",classical_univariate,96,96,results/predictions/etth1/classical_univariate...,1.478555e+02,11.557632,12.159584
1,etth1,mLSTM_reproduced,wavelet_learnable,96,96,results/predictions/etth1/mLSTM_reproduced_tes...,2.821726e+01,4.742370,5.311992
2,etth2,ETS,classical_univariate,96,96,results/predictions/etth2/classical_univariate...,4.480221e+01,5.540184,6.693446
3,etth2,mLSTM_reproduced,wavelet_learnable,96,96,results/predictions/etth2/mLSTM_reproduced_tes...,1.083756e+02,8.162314,10.410360
4,ettm1,"ARIMA(1, 0, 0)",classical_univariate,192,96,results/predictions/ettm1/classical_univariate...,1.477189e+02,11.549256,12.153965
5,ettm1,mLSTM_reproduced,wavelet_learnable,192,96,results/predictions/ettm1/mLSTM_reproduced_tes...,1.176870e+01,2.885064,3.430554
6,ettm2,"ARIMA(1, 1, 0)",classical_univariate,192,96,results/predictions/ettm2/classical_univariate...,7.615466e+01,7.267138,8.726663
7,ettm2,mLSTM_reproduced,wavelet_learnable,192,96,results/predictions/ettm2/mLSTM_reproduced_tes...,8.043043e+01,7.465128,8.968301
8,weather,"ARIMA(1, 0, 0)",classical_univariate,288,96,results/predictions/weather/classical_univaria...,8.521870e+02,24.246773,29.192242
9,weather,mLSTM_reproduced,wavelet_learnable,288,96,results/predictions/weather/mLSTM_reproduced_t...,4.822665e+02,16.498678,21.960567


In [2]:
# ============================================
# NEW FINAL CELL: Save raw-unit predictions for ARIMA/ETS and mLSTM_reproduced
# ============================================

import gc
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

ROOT = Path("/data/Sajjan_Singh/spml/wavelet_seq_project")
PROCESSED_ROOT = ROOT / "data" / "processed"
SPLIT_ROOT = ROOT / "data" / "splits"
PRED_ROOT = ROOT / "results" / "predictions"
TABLES_ROOT = ROOT / "results" / "tables"

PRED_ROOT.mkdir(parents=True, exist_ok=True)
TABLES_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

PHASEA_CFG = {
    "etth1": {"seq_len": 96, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 24},
    "etth2": {"seq_len": 96, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 24},
    "ettm1": {"seq_len": 192, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 96},
    "ettm2": {"seq_len": 192, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 96},
    "weather": {"seq_len": 288, "pred_len": 96, "batch_size": 192, "epochs": 25, "seasonality": 144},
    "electricity": {"seq_len": 168, "pred_len": 96, "batch_size": 128, "epochs": 25, "seasonality": 24},
    "traffic": {"seq_len": 168, "pred_len": 96, "batch_size": 96, "epochs": 25, "seasonality": 24},
    "exchange": {"seq_len": 96, "pred_len": 96, "batch_size": 256, "epochs": 25, "seasonality": 7},
    "ili": {"seq_len": 104, "pred_len": 24, "batch_size": 64, "epochs": 30, "seasonality": 52},
}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mse(y, yhat):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    return float(np.mean((y - yhat) ** 2))

def mae(y, yhat):
    y = np.asarray(y, dtype=np.float64)
    yhat = np.asarray(yhat, dtype=np.float64)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    return float(np.sqrt(mse(y, yhat)))

def load_bundle(dataset_name):
    npz_path = PROCESSED_ROOT / dataset_name / f"{dataset_name}_prepared.npz"
    split_path = SPLIT_ROOT / f"{dataset_name}_splits.json"
    arr = np.load(npz_path, allow_pickle=True)
    with open(split_path, "r") as f:
        splits = json.load(f)
    return {
        "values_raw": arr["values_raw"].astype(np.float32),
        "values_scaled": arr["values_scaled"].astype(np.float32),
        "target_idx": int(arr["target_idx"][0]),
        "scaler_mean": arr["scaler_mean"].astype(np.float32),
        "scaler_std": arr["scaler_std"].astype(np.float32),
        "splits": {k: int(v) for k, v in splits.items()},
    }

class ForecastWindowDataset(Dataset):
    def __init__(self, bundle, split_name, seq_len, pred_len):
        self.values_raw = bundle["values_raw"]
        self.values_scaled = bundle["values_scaled"]
        self.target_idx = bundle["target_idx"]
        s = bundle["splits"]

        if split_name == "train":
            b1, b2 = s["train_start"], s["train_end"]
        elif split_name == "val":
            b1, b2 = max(0, s["val_start"] - seq_len), s["val_end"]
        elif split_name == "test":
            b1, b2 = max(0, s["test_start"] - seq_len), s["test_end"]
        else:
            raise ValueError(split_name)

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.border1 = b1
        self.border2 = b2
        self.length = b2 - b1 - seq_len - pred_len + 1

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        s = self.border1 + idx
        x_scaled = self.values_scaled[s:s+self.seq_len]
        y_scaled = self.values_scaled[s+self.seq_len:s+self.seq_len+self.pred_len, self.target_idx]
        y_raw = self.values_raw[s+self.seq_len:s+self.seq_len+self.pred_len, self.target_idx]
        return {
            "x_scaled": torch.tensor(x_scaled, dtype=torch.float32),
            "y_scaled": torch.tensor(y_scaled, dtype=torch.float32),
            "y_raw": torch.tensor(y_raw, dtype=torch.float32),
        }

def make_loaders(bundle, seq_len, pred_len, batch_size):
    train_ds = ForecastWindowDataset(bundle, "train", seq_len, pred_len)
    val_ds = ForecastWindowDataset(bundle, "val", seq_len, pred_len)
    test_ds = ForecastWindowDataset(bundle, "test", seq_len, pred_len)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available()),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available()),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available()),
    )

class LearnableWaveletDecomp(nn.Module):
    def __init__(self, channels):
        super().__init__()
        sqrt2 = math.sqrt(2.0)
        lp = torch.tensor([1/sqrt2, 1/sqrt2], dtype=torch.float32)
        hp = torch.tensor([1/sqrt2, -1/sqrt2], dtype=torch.float32)

        self.low = nn.Conv1d(channels, channels, kernel_size=2, stride=2, groups=channels, bias=False)
        self.high = nn.Conv1d(channels, channels, kernel_size=2, stride=2, groups=channels, bias=False)

        with torch.no_grad():
            self.low.weight[:] = lp.view(1, 1, 2).repeat(channels, 1, 1)
            self.high.weight[:] = hp.view(1, 1, 2).repeat(channels, 1, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        if x.shape[-1] % 2 == 1:
            x = torch.cat([x, x[..., -1:]], dim=-1)

        low = self.low(x)
        high = self.high(x)

        low_up = low.repeat_interleave(2, dim=-1)[..., :x.shape[-1]]
        high_up = high.repeat_interleave(2, dim=-1)[..., :x.shape[-1]]

        return low_up.transpose(1, 2), high_up.transpose(1, 2)

class MLSTMBaseline(nn.Module):
    def __init__(self, input_dim, pred_len, hidden_dim=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.decomp = LearnableWaveletDecomp(input_dim)
        self.proj = nn.Linear(input_dim * 3, 128)
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, pred_len),
        )

    def forward(self, x):
        low, high = self.decomp(x)
        z = torch.cat([x, low, high], dim=-1)
        z = self.proj(z)
        out, _ = self.lstm(z)
        return self.head(out[:, -1, :])

@torch.no_grad()
def predict_raw(model, loader, target_mean, target_std):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        x = batch["x_scaled"].to(DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred_scaled = model(x)
        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
    return np.concatenate(preds, axis=0), np.concatenate(trues, axis=0)

def run_classical_and_save(dataset_name):
    bundle = load_bundle(dataset_name)
    cfg = PHASEA_CFG[dataset_name]
    s = bundle["splits"]
    target_idx = bundle["target_idx"]

    y = bundle["values_raw"][:, target_idx].astype(float)

    tr0, tr1 = s["train_start"], s["train_end"]
    va0, va1 = s["val_start"], s["val_end"]
    te0, te1 = s["test_start"], s["test_end"]

    y_train = y[tr0:tr1]
    y_val = y[va0:va1]
    y_test = y[te0:te1]

    best_name = None
    best_val_mse = np.inf
    best_test_pred = None

    try:
        sp = cfg["seasonality"]
        seasonal = "add" if len(y_train) > 2 * sp else None
        ets = ExponentialSmoothing(
            y_train,
            trend="add",
            seasonal=seasonal,
            seasonal_periods=sp if seasonal else None,
            initialization_method="estimated"
        ).fit(optimized=True)
        val_pred = np.asarray(ets.forecast(len(y_val)), dtype=float)
        val_m = mse(y_val, val_pred)

        trainval = y[:va1]
        ets2 = ExponentialSmoothing(
            trainval,
            trend="add",
            seasonal=seasonal,
            seasonal_periods=sp if seasonal else None,
            initialization_method="estimated"
        ).fit(optimized=True)
        test_pred = np.asarray(ets2.forecast(len(y_test)), dtype=float)

        if val_m < best_val_mse:
            best_val_mse = val_m
            best_name = "ETS"
            best_test_pred = test_pred
    except Exception as e:
        print(dataset_name, "ETS failed:", e)

    for order in [(1,0,0), (2,0,0), (1,1,0), (1,1,1), (2,1,2)]:
        try:
            ar = ARIMA(y_train, order=order).fit()
            val_pred = np.asarray(ar.forecast(len(y_val)), dtype=float)
            val_m = mse(y_val, val_pred)

            trainval = y[:va1]
            ar2 = ARIMA(trainval, order=order).fit()
            test_pred = np.asarray(ar2.forecast(len(y_test)), dtype=float)

            if val_m < best_val_mse:
                best_val_mse = val_m
                best_name = f"ARIMA{order}"
                best_test_pred = test_pred
        except Exception:
            pass

    if best_test_pred is None:
        raise RuntimeError(f"{dataset_name}: ARIMA/ETS all failed")

    pred_dir = PRED_ROOT / dataset_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    pred_path = pred_dir / "classical_univariate_test_predictions.npz"
    np.savez_compressed(
        pred_path,
        preds=best_test_pred.reshape(-1, 1),
        trues=y_test.reshape(-1, 1),
        seq_len=cfg["seq_len"],
        pred_len=cfg["pred_len"],
        model=best_name,
    )

    return {
        "dataset": dataset_name,
        "model": best_name,
        "family": "classical_univariate",
        "seq_len": cfg["seq_len"],
        "pred_len": cfg["pred_len"],
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": mse(y_test, best_test_pred),
        "test_mae": mae(y_test, best_test_pred),
        "test_rmse": rmse(y_test, best_test_pred),
    }

def run_mlstm_and_save(dataset_name):
    set_seed(42)
    bundle = load_bundle(dataset_name)
    cfg = PHASEA_CFG[dataset_name]

    seq_len = cfg["seq_len"]
    pred_len = cfg["pred_len"]
    batch_size = cfg["batch_size"]
    epochs = cfg["epochs"]

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    train_loader, val_loader, test_loader = make_loaders(bundle, seq_len, pred_len, batch_size)
    model = MLSTMBaseline(
        input_dim=bundle["values_scaled"].shape[1],
        pred_len=pred_len
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.MSELoss()

    best_val = np.inf
    patience = 5
    best_state = None

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            x = batch["x_scaled"].to(DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                pred = model(x)
                loss = criterion(pred, y)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        val_preds, val_trues = predict_raw(model, val_loader, target_mean, target_std)
        val_rmse = rmse(val_trues.reshape(-1), val_preds.reshape(-1))

        if val_rmse < best_val:
            best_val = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 5
        else:
            patience -= 1
            if patience == 0:
                break

    model.load_state_dict(best_state)

    test_preds, test_trues = predict_raw(model, test_loader, target_mean, target_std)

    pred_dir = PRED_ROOT / dataset_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    pred_path = pred_dir / "mLSTM_reproduced_test_predictions.npz"
    np.savez_compressed(
        pred_path,
        preds=test_preds,
        trues=test_trues,
        seq_len=seq_len,
        pred_len=pred_len,
    )

    row = {
        "dataset": dataset_name,
        "model": "mLSTM_reproduced",
        "family": "wavelet_learnable",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": mse(test_trues.reshape(-1), test_preds.reshape(-1)),
        "test_mae": mae(test_trues.reshape(-1), test_preds.reshape(-1)),
        "test_rmse": rmse(test_trues.reshape(-1), test_preds.reshape(-1)),
    }

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row

registry_rows = []

for ds in PHASEA_CFG:
    print("\n" + "=" * 100)
    print("DATASET:", ds)
    print("=" * 100)

    classical_row = run_classical_and_save(ds)
    print("Saved classical predictions:", classical_row["prediction_file"])
    registry_rows.append(classical_row)

    mlstm_row = run_mlstm_and_save(ds)
    print("Saved mLSTM predictions:", mlstm_row["prediction_file"])
    registry_rows.append(mlstm_row)

registry_df = pd.DataFrame(registry_rows)
registry_path = TABLES_ROOT / "phaseA_extra_prediction_registry.csv"
registry_df.to_csv(registry_path, index=False)

print("\nSaved:", registry_path)
display(registry_df)


DATASET: etth1


/home/sajjan/.conda/envs/myenv/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Saved classical predictions: results/predictions/etth1/classical_univariate_test_predictions.npz


/tmp/ipykernel_2175268/3140948556.py:308: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_2175268/3140948556.py:322: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_2175268/3140948556.py:186: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Saved mLSTM predictions: results/predictions/etth1/mLSTM_reproduced_test_predictions.npz

DATASET: etth2


...

/tmp/ipykernel_2175268/3140948556.py:308: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_2175268/3140948556.py:322: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_2175268/3140948556.py:186: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Saved mLSTM predictions: results/predictions/exchange/mLSTM_reproduced_test_predictions.npz

DATASET: ili
Saved classical predictions: results/predictions/ili/classical_univariate_test_predictions.npz


/tmp/ipykernel_2175268/3140948556.py:308: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
/tmp/ipykernel_2175268/3140948556.py:322: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/tmp/ipykernel_2175268/3140948556.py:186: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Saved mLSTM predictions: results/predictions/ili/mLSTM_reproduced_test_predictions.npz

Saved: /data/Sajjan_Singh/spml/wavelet_seq_project/results/tables/phaseA_extra_prediction_registry.csv


,dataset,model,family,seq_len,pred_len,prediction_file,test_mse,test_mae,test_rmse
0,etth1,"ARIMA(2, 0, 0)",classical_univariate,96,96,results/predictions/etth1/classical_univariate...,1.478555e+02,11.557632,12.159584
1,etth1,mLSTM_reproduced,wavelet_learnable,96,96,results/predictions/etth1/mLSTM_reproduced_tes...,2.821726e+01,4.742370,5.311992
2,etth2,ETS,classical_univariate,96,96,results/predictions/etth2/classical_univariate...,4.480221e+01,5.540184,6.693446
3,etth2,mLSTM_reproduced,wavelet_learnable,96,96,results/predictions/etth2/mLSTM_reproduced_tes...,1.083756e+02,8.162314,10.410360
4,ettm1,"ARIMA(1, 0, 0)",classical_univariate,192,96,results/predictions/ettm1/classical_univariate...,1.477189e+02,11.549256,12.153965
5,ettm1,mLSTM_reproduced,wavelet_learnable,192,96,results/predictions/ettm1/mLSTM_reproduced_tes...,1.176870e+01,2.885064,3.430554
6,ettm2,"ARIMA(1, 1, 0)",classical_univariate,192,96,results/predictions/ettm2/classical_univariate...,7.615466e+01,7.267138,8.726663
7,ettm2,mLSTM_reproduced,wavelet_learnable,192,96,results/predictions/ettm2/mLSTM_reproduced_tes...,8.043043e+01,7.465128,8.968301
8,weather,"ARIMA(1, 0, 0)",classical_univariate,288,96,results/predictions/weather/classical_univaria...,8.521870e+02,24.246773,29.192242
9,weather,mLSTM_reproduced,wavelet_learnable,288,96,results/predictions/weather/mLSTM_reproduced_t...,4.822665e+02,16.498678,21.960567
